# AutoDDG Pipeline — Scalable Dataset Description Generation
**Team DataScribes** | Rishabh Patil 

This notebook implements an end-to-end Data Engineering pipeline for automatically generating high-quality textual descriptions for open datasets using **Apache Spark** on the NYU HPC DataProc cluster and **GPT-4o-mini** via the OpenAI API.

## Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Configuration & Environment Setup |
| 2 | Library Imports |
| 3 | Spark Session Initialization |
| 4 | Step 1 — Metadata Ingestion |
| 5 | Step 2 — CSV Sample Fetching |
| 6 | Step 3 — Data Preparation |
| 7 | Step 4 — LLM Description Generation |
| 8 | Step 5 — Evaluation Table |
| 9 | Evaluation — Text Similarity Metrics |
| 10 | Evaluation — Retrieval (NDCG@10) |
| 11 | Evaluation — BERTScore |
| 12 | Qualitative Evaluation |

---
## Step 1 — Configuration

All pipeline parameters are defined here. Edit these values before running the notebook.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `NYC_LIMIT` | 200 | Datasets to fetch from NYC Open Data |
| `DATA_GOV_LIMIT` | 200 | Datasets to fetch from Data.gov |
| `MODEL_NAME` | gpt-4o-mini | LLM used for description generation |
| `MAX_SAMPLE_ROWS` | 20 | CSV rows sampled per dataset |
| `REQUEST_TIMEOUT` | 40s | HTTP timeout for CSV downloads |
| `MAX_ELAPSED_SECONDS` | 60s | Max streaming time per dataset |
| `MAX_CHARS` | 12000 | Max characters sent to LLM |
| `MAX_LINES` | 20 | Max lines sent to LLM |

All intermediate outputs are stored in HDFS under the user's directory on the DataProc cluster.

In [1]:
NYC_LIMIT       = 200   # datasets from NYC Open Data
DATA_GOV_LIMIT  = 200    # datasets from Data.gov

HDFS_BASE = "hdfs:///user/rbp5812_nyu_edu/pipeline"
LOCAL_OUTPUT_PARQUET = "/home/rbp5812_nyu_edu/outputs/descriptions.parquet"

MODEL_NAME          = "gpt-4o-mini"

MAX_SAMPLE_ROWS     = 20
REQUEST_TIMEOUT     = 40
MAX_ELAPSED_SECONDS = 60
MAX_CHARS           = 12000  # was 8000
MAX_LINES           = 20  

# Derived HDFS paths
HDFS_METADATA = f"{HDFS_BASE}/step1_metadata"
HDFS_SAMPLES  = f"{HDFS_BASE}/step2_samples"
HDFS_PREPARED = f"{HDFS_BASE}/step3_prepared"
HDFS_EVAL     = f"{HDFS_BASE}/step5_evaluation"

print(f"NYC datasets      : {NYC_LIMIT}")
print(f"Data.gov datasets : {DATA_GOV_LIMIT}")
print(f"Total requested   : {NYC_LIMIT + DATA_GOV_LIMIT}")
print(f"Model             : {MODEL_NAME}")
print(f"HDFS base         : {HDFS_BASE}")
print(f"Local output      : {LOCAL_OUTPUT_PARQUET}")


NYC datasets      : 200
Data.gov datasets : 200
Total requested   : 400
Model             : gpt-4o-mini
HDFS base         : hdfs:///user/rbp5812_nyu_edu/pipeline
Local output      : /home/rbp5812_nyu_edu/outputs/descriptions.parquet


---
## Step 2 — Library Imports

We import standard Python libraries alongside:
- **`pandas`** — local DataFrame manipulation
- **`requests`** — HTTP calls to open data APIs
- **`openai`** — GPT-4o-mini API client
- **`pyspark`** — distributed processing on the DataProc cluster
- **`autoddg`** — the AutoDDG framework for description generation

In [2]:
from __future__ import annotations

import csv
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from openai import OpenAI
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from autoddg import AutoDDG

csv.field_size_limit(sys.maxsize)
print("Imports OK.")


Imports OK.


---
## Step 3 — Spark Session Initialization

We initialize a Spark session connected to the **YARN** cluster manager on NYU DataProc. The `--deploy-mode client` flag ensures the driver runs on the master node, which is required for interactive Jupyter execution.

The cluster consists of:
- 1 master node (`nyu-dataproc-m`) — n1-standard-32
- 2 worker nodes (`nyu-dataproc-w-0`, `nyu-dataproc-w-1`) — n1-standard-16 each
- 1 preemptible secondary worker — n1-standard-16

In [3]:
import os

# Remove conflicting deploy mode setting
os.environ.pop("SPARK_SUBMIT_OPTS", None)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master yarn --deploy-mode client pyspark-shell"

spark = (
    SparkSession.builder
    .appName("autoddg_pipeline")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
print(spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 06:38:31 INFO SparkEnv: Registering MapOutputTracker
26/05/07 06:38:31 INFO SparkEnv: Registering BlockManagerMaster
26/05/07 06:38:31 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/07 06:38:31 INFO SparkEnv: Registering OutputCommitCoordinator


---
## Step 4 — Metadata Ingestion

### Data Sources
We collect dataset metadata from two open data repositories:

**NYC Open Data** (Socrata API)
- Catalog endpoint returns a list of all datasets
- For each dataset we fetch detailed metadata including column names, types, and license info
- Download URLs are constructed using the Socrata CSV export format

**Data.gov** (CKAN Catalog API)
- Search endpoint returns dataset records in DCAT format
- We extract title, description, keywords, and CSV download links from the `dcat.distribution` field
- Many Data.gov datasets do not have direct CSV links — these are handled gracefully

### Schema
All metadata is normalized into a unified Spark schema with 14 fields and written to HDFS as Parquet for efficient downstream processing.

In [4]:
NYC_CATALOG_URL     = "https://data.cityofnewyork.us/api/views.json"
NYC_VIEW_URL_TMPL   = "https://data.cityofnewyork.us/api/views/{dataset_id}.json"
DATA_GOV_SEARCH_URL = "https://catalog.data.gov/search"

#original
def safe_get(dct: Dict[str, Any], key: str, default=None):
    return dct[key] if key in dct else default


def fetch_json(
    url: str,
    params: Optional[Dict[str, Any]] = None,
    retries: int = 3,
    sleep_seconds: float = 2.0,
) -> Any:
    last_error: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                url, params=params, timeout=60,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_error = e
            print(f"[WARN] fetch failed attempt={attempt}/{retries} url={url}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)
    raise RuntimeError(f"Failed to fetch URL after {retries} attempts: {url}") from last_error


def extract_keywords(category: Optional[str], tags: Optional[List[str]]) -> List[str]:
    out: List[str] = []
    if category:
        out.append(str(category))
    if tags:
        out.extend([str(t) for t in tags if t is not None])
    seen = set()
    deduped: List[str] = []
    for x in out:
        if x not in seen:
            seen.add(x)
            deduped.append(x)
    return deduped


def extract_column_names(columns: Optional[List[Dict[str, Any]]]) -> List[str]:
    names: List[str] = []
    for col in columns or []:
        name = safe_get(col, "name")
        if name:
            names.append(str(name))
    return names


def extract_column_types(columns: Optional[List[Dict[str, Any]]]) -> List[str]:
    types_: List[str] = []
    for col in columns or []:
        dtype = safe_get(col, "dataTypeName")
        types_.append(str(dtype) if dtype is not None else "unknown")
    return types_


def normalize_nyc_dataset(summary_entry: Dict[str, Any], detail_entry: Dict[str, Any]) -> Dict[str, Any]:
    dataset_id      = safe_get(summary_entry, "id")
    title           = safe_get(summary_entry, "name")
    description     = safe_get(summary_entry, "description")
    category        = safe_get(summary_entry, "category")
    tags            = safe_get(summary_entry, "tags", [])
    rows_updated_at = safe_get(summary_entry, "rowsUpdatedAt")
    detail_columns  = safe_get(detail_entry, "columns", [])
    license_info    = safe_get(detail_entry, "license")
    license_name    = None
    if isinstance(license_info, dict):
        license_name = safe_get(license_info, "name")
    return {
        "dataset_id":            str(dataset_id) if dataset_id is not None else None,
        "source":                "nyc_open_data",
        "title":                 str(title) if title is not None else None,
        "original_description":  str(description) if description is not None else None,
        "keywords_json":         json.dumps(extract_keywords(category, tags), ensure_ascii=False),
        "column_names_json":     json.dumps(extract_column_names(detail_columns), ensure_ascii=False),
        "column_types_raw_json": json.dumps(extract_column_types(detail_columns), ensure_ascii=False),
        "download_url":          f"https://data.cityofnewyork.us/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD" if dataset_id else None,
        "landing_page_url":      f"https://data.cityofnewyork.us/d/{dataset_id}" if dataset_id else None,
        "record_count_estimate": None,
        "last_updated":          str(rows_updated_at) if rows_updated_at is not None else None,
        "license":               str(license_name) if license_name is not None else None,
        "sample_rows_json":      None,
        "raw_metadata_json":     json.dumps(
            {"summary_entry": summary_entry, "detail_entry": detail_entry},
            ensure_ascii=False),
    }


def fetch_nyc_metadata(limit: int) -> List[Dict[str, Any]]:
    payload = fetch_json(NYC_CATALOG_URL)
    if not isinstance(payload, list):
        raise ValueError("Expected NYC catalog response to be a list.")
    trimmed = payload[:limit]
    rows: List[Dict[str, Any]] = []
    for idx, entry in enumerate(trimmed, start=1):
        dataset_id = safe_get(entry, "id")
        if not dataset_id:
            continue
        print(f"[INFO] NYC detail {idx}/{len(trimmed)} dataset_id={dataset_id}")
        detail = fetch_json(NYC_VIEW_URL_TMPL.format(dataset_id=dataset_id))
        rows.append(normalize_nyc_dataset(entry, detail))
    return rows


def normalize_data_gov_dataset(entry: Dict[str, Any]) -> Dict[str, Any]:
    dcat         = entry.get("dcat", {}) or {}
    dataset_id   = entry.get("identifier") or dcat.get("identifier")
    title        = entry.get("title") or dcat.get("title")
    notes        = entry.get("description") or dcat.get("description")
    tags         = entry.get("keyword") or dcat.get("keyword") or []
    modified     = dcat.get("modified")
    license_url  = dcat.get("license")
    landing_page = dcat.get("landingPage")

    distributions = dcat.get("distribution", []) or []
    csv_url = None
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        access_url = dist.get("accessURL") or dist.get("downloadURL")
        media_type = (dist.get("mediaType") or "").lower()
        fmt        = (dist.get("format") or "").lower()
        if access_url and (
            "csv" in media_type or "csv" in fmt
            or str(access_url).lower().endswith(".csv")
        ):
            csv_url = access_url
            break

    return {
        "dataset_id":            str(dataset_id) if dataset_id is not None else None,
        "source":                "data_gov",
        "title":                 str(title) if title is not None else None,
        "original_description":  str(notes) if notes is not None else None,
        "keywords_json":         json.dumps(
            [str(x) for x in tags] if isinstance(tags, list) else [], ensure_ascii=False),
        "column_names_json":     json.dumps([], ensure_ascii=False),
        "column_types_raw_json": json.dumps([], ensure_ascii=False),
        "download_url":          csv_url,
        "landing_page_url":      landing_page,
        "record_count_estimate": None,
        "last_updated":          str(modified) if modified is not None else None,
        "license":               str(license_url) if license_url is not None else None,
        "sample_rows_json":      None,
        "raw_metadata_json":     json.dumps(entry, ensure_ascii=False),
    }


def fetch_data_gov_metadata(limit: int) -> List[Dict[str, Any]]:
    params  = {"q": "", "per_page": limit}
    payload = fetch_json(DATA_GOV_SEARCH_URL, params=params)
    results = payload.get("results", [])
    if not isinstance(results, list):
        raise ValueError(f"Unexpected Data.gov response shape — 'results' is {type(results)}")
    print(f"[INFO] Data.gov returned {len(results)} entries")
    return [normalize_data_gov_dataset(entry) for entry in results[:limit]]


def build_metadata_schema() -> T.StructType:
    return T.StructType([
        T.StructField("dataset_id",            T.StringType(), True),
        T.StructField("source",                T.StringType(), True),
        T.StructField("title",                 T.StringType(), True),
        T.StructField("original_description",  T.StringType(), True),
        T.StructField("keywords_json",         T.StringType(), True),
        T.StructField("column_names_json",     T.StringType(), True),
        T.StructField("column_types_raw_json", T.StringType(), True),
        T.StructField("download_url",          T.StringType(), True),
        T.StructField("landing_page_url",      T.StringType(), True),
        T.StructField("record_count_estimate", T.LongType(),   True),
        T.StructField("last_updated",          T.StringType(), True),
        T.StructField("license",               T.StringType(), True),
        T.StructField("sample_rows_json",      T.StringType(), True),
        T.StructField("raw_metadata_json",     T.StringType(), True),
    ])


### Fetching Metadata
We now call both APIs and combine the results into a single Spark DataFrame, then write it to HDFS at `step1_metadata`.

In [5]:
print(f"[INFO] Fetching NYC metadata: {NYC_LIMIT}")
nyc_rows = fetch_nyc_metadata(limit=NYC_LIMIT)

print(f"[INFO] Fetching Data.gov metadata: {DATA_GOV_LIMIT}")
data_gov_rows = fetch_data_gov_metadata(limit=DATA_GOV_LIMIT)

all_rows = nyc_rows + data_gov_rows
if not all_rows:
    raise ValueError("No metadata rows were fetched from either source.")

metadata_df = spark.createDataFrame(all_rows, schema=build_metadata_schema())

print(f"[INFO] Writing {metadata_df.count()} combined rows to {HDFS_METADATA}")
metadata_df.write.mode("overwrite").parquet(HDFS_METADATA)

metadata_df.groupBy("source").count().show(truncate=False)
metadata_df.select("dataset_id", "source", "title", "download_url").show(30, truncate=False)


[INFO] Fetching NYC metadata: 200
[INFO] NYC detail 1/200 dataset_id=qhkz-4dqm
[INFO] NYC detail 2/200 dataset_id=wgnh-qwsg
[INFO] NYC detail 3/200 dataset_id=naav-ygga
[INFO] NYC detail 4/200 dataset_id=5mb3-padx
[INFO] NYC detail 5/200 dataset_id=i2im-iqtt
[INFO] NYC detail 6/200 dataset_id=gdk4-mbsv
[INFO] NYC detail 7/200 dataset_id=pztn-9bne
[INFO] NYC detail 8/200 dataset_id=5ucz-vwe8
[INFO] NYC detail 9/200 dataset_id=m5vz-tzqv
[INFO] NYC detail 10/200 dataset_id=8zf9-spf8
[INFO] NYC detail 11/200 dataset_id=wh8n-imgd
[INFO] NYC detail 12/200 dataset_id=ct66-47at
[INFO] NYC detail 13/200 dataset_id=6up2-gnw8
[INFO] NYC detail 14/200 dataset_id=76ig-c548
[INFO] NYC detail 15/200 dataset_id=rixx-fc37
[INFO] NYC detail 16/200 dataset_id=aq7i-eu5q
[INFO] NYC detail 17/200 dataset_id=ag7h-2pg6
[INFO] NYC detail 18/200 dataset_id=kb2e-tjy3
[INFO] NYC detail 19/200 dataset_id=6ztr-wgff
[INFO] NYC detail 20/200 dataset_id=vhqf-adkz
[INFO] NYC detail 21/200 dataset_id=kbgp-72qi
[INFO] NY

[INFO] NYC detail 178/200 dataset_id=w462-digi
[INFO] NYC detail 179/200 dataset_id=tdej-swyi
[INFO] NYC detail 180/200 dataset_id=k462-uqyk
[INFO] NYC detail 181/200 dataset_id=bzg2-2abf
[INFO] NYC detail 182/200 dataset_id=592z-n7dk
[INFO] NYC detail 183/200 dataset_id=2ync-kihj
[INFO] NYC detail 184/200 dataset_id=2er2-jqsx
[INFO] NYC detail 185/200 dataset_id=gtdx-4w36
[INFO] NYC detail 186/200 dataset_id=n47m-7kn5
[INFO] NYC detail 187/200 dataset_id=gq2c-wem9
[INFO] NYC detail 188/200 dataset_id=j62s-m9yf
[INFO] NYC detail 189/200 dataset_id=shr7-eqdc
[INFO] NYC detail 190/200 dataset_id=fcnc-95cn
[INFO] NYC detail 191/200 dataset_id=tmt9-43em
[INFO] NYC detail 192/200 dataset_id=53jq-yvwd
[INFO] NYC detail 193/200 dataset_id=sm2x-35i7
[INFO] NYC detail 194/200 dataset_id=mtj6-vmci
[INFO] NYC detail 195/200 dataset_id=6ax4-q5k4
[INFO] NYC detail 196/200 dataset_id=hjz2-y62k
[INFO] NYC detail 197/200 dataset_id=fkec-mjr6
[INFO] NYC detail 198/200 dataset_id=sstf-x9es
[INFO] NYC de

26/05/07 03:58:01 WARN TaskSetManager: Stage 0 contains a task of very large size (4669 KiB). The maximum recommended task size is 1000 KiB.


[INFO] Writing 400 combined rows to hdfs:///user/rbp5812_nyu_edu/pipeline/step1_metadata


26/05/07 04:00:29 WARN TaskSetManager: Stage 3 contains a task of very large size (4669 KiB). The maximum recommended task size is 1000 KiB.
26/05/07 04:00:40 WARN TaskSetManager: Stage 4 contains a task of very large size (4669 KiB). The maximum recommended task size is 1000 KiB.


+-------------+-----+
|source       |count|
+-------------+-----+
|nyc_open_data|200  |
|data_gov     |200  |
+-------------+-----+



26/05/07 04:00:44 WARN TaskSetManager: Stage 7 contains a task of very large size (4669 KiB). The maximum recommended task size is 1000 KiB.


+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------+
|dataset_id|source       |title                                                                                              |download_url                                                                  |
+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                            |https://data.cityofnewyork.us/api/views/qhkz-4dqm/rows.csv?accessType=DOWNLOAD|
|wgnh-qwsg |nyc_open_data|Citywide Mobility Survey - Trip 2024                                                               |https://data.cityofnewyork.us/api/views/wgnh-qwsg/

---
## Step 5 — CSV Sample Fetching

For each dataset that has a valid download URL, we stream the first `MAX_SAMPLE_ROWS` rows of the CSV file. This avoids downloading entire datasets which can be gigabytes in size.

**Key design decisions:**
- **Streaming**: We use `response.iter_lines()` to read line by line, stopping after enough rows
- **Timeout guard**: A wall-clock timer stops fetching if `MAX_ELAPSED_SECONDS` is exceeded
- **Retry logic**: Up to 3 attempts per dataset with 2-second backoff
- **Graceful failure**: Datasets that fail (missing URL, 403, timeout) are marked as `error` and excluded from later steps — the pipeline continues

Results are written to HDFS at `step2_samples`.

In [6]:
def fetch_csv_sample_text(
    download_url: str,
    max_rows: int = 5,
    retries: int = 3,
    sleep_seconds: float = 2.0,
    request_timeout: int = 20,
    max_elapsed_seconds: int = 30,
) -> Optional[str]:
    last_error: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        start_time = time.time()
        try:
            with requests.get(
                download_url,
                timeout=request_timeout,
                stream=True,
                headers={"User-Agent": "Mozilla/5.0"},
            ) as response:
                response.raise_for_status()
                lines: List[str] = []
                for line in response.iter_lines(decode_unicode=True):
                    if time.time() - start_time > max_elapsed_seconds:
                        raise TimeoutError(f"sample fetch exceeded {max_elapsed_seconds}s")
                    if line is None:
                        continue
                    line = line.strip()
                    if not line:
                        continue
                    lines.append(line)
                    if len(lines) >= max_rows + 1:
                        break
                if not lines:
                    return None
                reader    = csv.reader(io.StringIO("\n".join(lines)))
                rows      = list(reader)
                if not rows:
                    return None
                header    = rows[0]
                data_rows = rows[1:]
                output    = io.StringIO()
                writer    = csv.writer(output, lineterminator="\n")
                writer.writerow(header)
                writer.writerows(data_rows)
                sample_csv = output.getvalue().strip()
                return sample_csv if sample_csv else None
        except Exception as e:
            last_error = e
            print(f"[WARN] sample fetch failed attempt={attempt}/{retries} url={download_url}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)
    print(f"[ERROR] giving up on url={download_url}: {last_error}")
    return None


def build_samples_schema() -> T.StructType:
    return T.StructType([
        T.StructField("dataset_id",           T.StringType(), True),
        T.StructField("source",               T.StringType(), True),
        T.StructField("title",                T.StringType(), True),
        T.StructField("original_description", T.StringType(), True),
        T.StructField("download_url",         T.StringType(), True),
        T.StructField("landing_page_url",     T.StringType(), True),
        T.StructField("sample_csv",           T.StringType(), True),
        T.StructField("sample_fetch_status",  T.StringType(), True),
        T.StructField("error_message",        T.StringType(), True),
    ])


def normalize_output_rows(output_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    required_cols = [
        "dataset_id", "source", "title", "original_description",
        "download_url", "landing_page_url", "sample_csv",
        "sample_fetch_status", "error_message",
    ]
    out_pd = pd.DataFrame(output_rows)
    for col in required_cols:
        if col not in out_pd.columns:
            out_pd[col] = None
    out_pd = out_pd[required_cols]
    for col in required_cols:
        out_pd[col] = out_pd[col].where(pd.notnull(out_pd[col]), None)
    return out_pd


In [7]:
step1_df = spark.read.parquet(HDFS_METADATA)

metadata_pd = (
    step1_df
    .select("dataset_id", "source", "title", "original_description",
            "download_url", "landing_page_url")
    .toPandas()
)

output_rows: List[Dict[str, Any]] = []
total = len(metadata_pd)
print(f"[INFO] Total metadata rows to process: {total}")

for idx, row in metadata_pd.iterrows():
    dataset_id           = row.get("dataset_id")
    source               = row.get("source")
    title                = row.get("title")
    original_description = row.get("original_description")
    download_url         = row.get("download_url")
    landing_page_url     = row.get("landing_page_url")

    print(f"[INFO] Fetching sample {idx + 1}/{total} for source={source} dataset_id={dataset_id}")

    sample_csv    = None
    error_message = None
    status        = "error"

    try:
        if download_url is None or not str(download_url).strip():
            raise ValueError("missing_download_url")
        sample_csv = fetch_csv_sample_text(
            download_url=str(download_url),
            max_rows=MAX_SAMPLE_ROWS,
            request_timeout=REQUEST_TIMEOUT,
            max_elapsed_seconds=MAX_ELAPSED_SECONDS,
        )
        if sample_csv is None or not str(sample_csv).strip():
            raise ValueError("empty_or_unreadable_sample")
        status = "success"
    except Exception as e:
        error_message = str(e)
        print(f"[ERROR] source={source} dataset_id={dataset_id} error={error_message}")

    output_rows.append({
        "dataset_id":           None if dataset_id is None else str(dataset_id),
        "source":               None if source is None else str(source),
        "title":                None if title is None else str(title),
        "original_description": None if original_description is None else str(original_description),
        "download_url":         None if download_url is None else str(download_url),
        "landing_page_url":     None if landing_page_url is None else str(landing_page_url),
        "sample_csv":           sample_csv,
        "sample_fetch_status":  status,
        "error_message":        error_message,
    })

if not output_rows:
    raise ValueError("No output rows were produced.")

out_pd        = normalize_output_rows(output_rows)
samples_spark = spark.createDataFrame(out_pd, schema=build_samples_schema())
samples_spark.write.mode("overwrite").parquet(HDFS_SAMPLES)

print(f"[INFO] Wrote {len(out_pd)} sampled datasets to {HDFS_SAMPLES}")
samples_spark.groupBy("source", "sample_fetch_status").count().show(truncate=False)
samples_spark.select(
    "dataset_id", "source", "title", "sample_fetch_status", "error_message"
).show(100, truncate=False)


[INFO] Total metadata rows to process: 400
[INFO] Fetching sample 1/400 for source=nyc_open_data dataset_id=qhkz-4dqm
[INFO] Fetching sample 2/400 for source=nyc_open_data dataset_id=wgnh-qwsg
[INFO] Fetching sample 3/400 for source=nyc_open_data dataset_id=naav-ygga
[INFO] Fetching sample 4/400 for source=nyc_open_data dataset_id=5mb3-padx
[INFO] Fetching sample 5/400 for source=nyc_open_data dataset_id=i2im-iqtt
[INFO] Fetching sample 6/400 for source=nyc_open_data dataset_id=gdk4-mbsv
[INFO] Fetching sample 7/400 for source=nyc_open_data dataset_id=pztn-9bne
[INFO] Fetching sample 8/400 for source=nyc_open_data dataset_id=5ucz-vwe8
[INFO] Fetching sample 9/400 for source=nyc_open_data dataset_id=m5vz-tzqv
[INFO] Fetching sample 10/400 for source=nyc_open_data dataset_id=8zf9-spf8
[INFO] Fetching sample 11/400 for source=nyc_open_data dataset_id=wh8n-imgd
[INFO] Fetching sample 12/400 for source=nyc_open_data dataset_id=ct66-47at
[INFO] Fetching sample 13/400 for source=nyc_open_data

[INFO] Fetching sample 77/400 for source=nyc_open_data dataset_id=fm4z-qud6
[INFO] Fetching sample 78/400 for source=nyc_open_data dataset_id=n7f2-dyvt
[INFO] Fetching sample 79/400 for source=nyc_open_data dataset_id=dwrg-kzni
[INFO] Fetching sample 80/400 for source=nyc_open_data dataset_id=kwss-yksz
[INFO] Fetching sample 81/400 for source=nyc_open_data dataset_id=wyj6-frpa
[INFO] Fetching sample 82/400 for source=nyc_open_data dataset_id=kizp-4dfk
[INFO] Fetching sample 83/400 for source=nyc_open_data dataset_id=2c5m-rke8
[ERROR] source=nyc_open_data dataset_id=2c5m-rke8 error=empty_or_unreadable_sample
[INFO] Fetching sample 84/400 for source=nyc_open_data dataset_id=2w2g-fk3i
[INFO] Fetching sample 85/400 for source=nyc_open_data dataset_id=2juy-aj8e
[INFO] Fetching sample 86/400 for source=nyc_open_data dataset_id=yqww-f9f3
[ERROR] source=nyc_open_data dataset_id=yqww-f9f3 error=empty_or_unreadable_sample
[INFO] Fetching sample 87/400 for source=nyc_open_data dataset_id=53n2-m85

[INFO] Fetching sample 142/400 for source=nyc_open_data dataset_id=dt48-h9kx
[ERROR] source=nyc_open_data dataset_id=dt48-h9kx error=empty_or_unreadable_sample
[INFO] Fetching sample 143/400 for source=nyc_open_data dataset_id=vhcs-h6rv
[ERROR] source=nyc_open_data dataset_id=vhcs-h6rv error=empty_or_unreadable_sample
[INFO] Fetching sample 144/400 for source=nyc_open_data dataset_id=akq4-haa2
[INFO] Fetching sample 145/400 for source=nyc_open_data dataset_id=ku5s-ingk
[INFO] Fetching sample 146/400 for source=nyc_open_data dataset_id=mhst-xhix
[ERROR] source=nyc_open_data dataset_id=mhst-xhix error=empty_or_unreadable_sample
[INFO] Fetching sample 147/400 for source=nyc_open_data dataset_id=ce23-cck4
[INFO] Fetching sample 148/400 for source=nyc_open_data dataset_id=484j-8mzq
[ERROR] source=nyc_open_data dataset_id=484j-8mzq error=empty_or_unreadable_sample
[INFO] Fetching sample 149/400 for source=nyc_open_data dataset_id=bhci-bpwh
[INFO] Fetching sample 150/400 for source=nyc_open_d

[WARN] sample fetch failed attempt=2/3 url=https://www.usitc.gov/sites/default/files/tata/hts/hts_2024_basic_edition_csv.csv: 403 Client Error: Forbidden for url: https://www.usitc.gov/sites/default/files/tata/hts/hts_2024_basic_edition_csv.csv
[WARN] sample fetch failed attempt=3/3 url=https://www.usitc.gov/sites/default/files/tata/hts/hts_2024_basic_edition_csv.csv: 403 Client Error: Forbidden for url: https://www.usitc.gov/sites/default/files/tata/hts/hts_2024_basic_edition_csv.csv
[ERROR] giving up on url=https://www.usitc.gov/sites/default/files/tata/hts/hts_2024_basic_edition_csv.csv: 403 Client Error: Forbidden for url: https://www.usitc.gov/sites/default/files/tata/hts/hts_2024_basic_edition_csv.csv
[ERROR] source=data_gov dataset_id=https://hts.usitc.gov/download?release=2024HTSBasic&releaseDate=12%2F11%2F2023 error=empty_or_unreadable_sample
[INFO] Fetching sample 210/400 for source=data_gov dataset_id=USDA-ERS-00071
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL

[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data_gov dataset_id=GSA-4495 error=empty_or_unreadable_sample
[INFO] Fetching sample 229/400 for source=data_gov dataset_id=https://data.transportation.gov/api/views/56fa-sf82
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data

...

[INFO] Fetching sample 382/400 for source=data_gov dataset_id=FHFA8590
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data_gov dataset_id=FHFA8590 error=empty_or_unreadable_sample
[INFO] Fetching sample 383/400 for source=data_gov dataset_id=2b7ec03d-67dc-4544-b1d6-b9aa02f693c6
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'na

26/05/07 04:31:48 WARN TaskSetManager: Stage 10 contains a task of very large size (27299 KiB). The maximum recommended task size is 1000 KiB.


[INFO] Wrote 400 sampled datasets to hdfs:///user/rbp5812_nyu_edu/pipeline/step2_samples


26/05/07 04:31:49 WARN TaskSetManager: Stage 11 contains a task of very large size (27299 KiB). The maximum recommended task size is 1000 KiB.
26/05/07 04:31:50 WARN TaskSetManager: Stage 14 contains a task of very large size (27299 KiB). The maximum recommended task size is 1000 KiB.


+-------------+-------------------+-----+
|source       |sample_fetch_status|count|
+-------------+-------------------+-----+
|data_gov     |success            |74   |
|data_gov     |error              |126  |
|nyc_open_data|success            |162  |
|nyc_open_data|error              |38   |
+-------------+-------------------+-----+

+----------+-------------+--------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|dataset_id|source       |title                                                                                                   |sample_fetch_status|error_message             |
+----------+-------------+--------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                             

---
## Step 6 — Data Preparation & Filtering

We filter the sampled dataset to keep only records where:
1. `sample_fetch_status == "success"` — CSV was successfully downloaded
2. `sample_csv` is not null or empty

This produces a clean Parquet file at `step3_prepared` containing only datasets ready for LLM description generation.

In [8]:
step2_df = spark.read.parquet(HDFS_SAMPLES)

prepared_df = (
    step2_df
    .filter(F.col("sample_fetch_status") == "success")
    .filter(F.col("sample_csv").isNotNull())
    .filter(F.trim(F.col("sample_csv")) != "")
    .select(
        F.col("dataset_id").cast("string").alias("dataset_id"),
        F.col("title").cast("string").alias("title"),
        F.col("source").cast("string").alias("source"),
        F.col("sample_csv").cast("string").alias("sample_csv"),
        F.col("original_description").cast("string").alias("original_description"),
        F.col("download_url").cast("string").alias("download_url"),
        F.col("landing_page_url").cast("string").alias("landing_page_url"),
    )
)

prepared_df.write.mode("overwrite").parquet(HDFS_PREPARED)

print("[INFO] Prepared records written.")
print(f"[INFO] Output path: {HDFS_PREPARED}")
print(f"[INFO] Prepared row count: {prepared_df.count()}")
prepared_df.groupBy("source").count().show(truncate=False)
prepared_df.select("dataset_id", "source", "title").show(100, truncate=False)


[INFO] Prepared records written.
[INFO] Output path: hdfs:///user/rbp5812_nyu_edu/pipeline/step3_prepared


[INFO] Prepared row count: 236


+-------------+-----+
|source       |count|
+-------------+-----+
|nyc_open_data|162  |
|data_gov     |74   |
+-------------+-----+



+----------+-------------+--------------------------------------------------------------------------------------------------------+
|dataset_id|source       |title                                                                                                   |
+----------+-------------+--------------------------------------------------------------------------------------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                                 |
|wgnh-qwsg |nyc_open_data|Citywide Mobility Survey - Trip 2024                                                                    |
|naav-ygga |nyc_open_data|Citywide Mobility Survey - Day 2024                                                                     |
|5mb3-padx |nyc_open_data|Citywide Mobility Survey - Household 2024                                                               |
|i2im-iqtt |nyc_open_data|Citywide Mobility Survey - Person 2024            

### Helper Functions
Utility functions for copying HDFS parquet files to local storage and truncating large CSV samples before sending to the LLM.

In [9]:
def run_cmd(cmd: List[str]) -> None:
    print(f"[CMD] {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


def shrink_sample_csv(
    sample_csv: Optional[str],
    max_chars: int = 8000,
    max_lines: int = 8,
) -> Optional[str]:
    if sample_csv is None:
        return None
    text = str(sample_csv).strip()
    if not text:
        return None
    lines      = text.splitlines()
    if not lines:
        return None
    header     = lines[0]
    data_lines = lines[1 : 1 + max_lines]
    shrunk     = "\n".join([header] + data_lines)
    if len(shrunk) > max_chars:
        shrunk = shrunk[:max_chars]
    return shrunk.strip() if shrunk.strip() else None


def copy_hdfs_parquet_to_local(hdfs_path: str, local_parent_dir: str) -> str:
    local_path = os.path.join(local_parent_dir, Path(hdfs_path).name)
    if os.path.exists(local_path):
        shutil.rmtree(local_path, ignore_errors=True)
    run_cmd(["hdfs", "dfs", "-get", hdfs_path, local_parent_dir])
    return local_path


## Step 7 — LLM Description Generation (Full AutoDDG Pipeline)

For each prepared dataset we run the **complete AutoDDG pipeline** — 5 stages per dataset:

| Stage | Method | What it does | LLM cost |
|---|---|---|---|
| A | `shrink_sample_csv()` | Truncate CSV to MAX_CHARS / MAX_LINES | None |
| B | `pd.read_csv()` | Parse CSV string → pandas DataFrame | None |
| C | `profile_dataframe()` | Structural stats via datamart-profiler (missing values, distributions, data types) | None |
| D | `analyze_semantics()` | LLM infers what each column *means* semantically | 1 API call |
| E | `generate_topic()` | Uses title + original description + CSV → 2–3 word topic | 1 API call |
| F | `describe_dataset()` | Full description using all context: profile + semantics + topic + CSV | 1 API call |
| G | `expand_description_for_search()` | Keyword-rich search-optimised variant of the description | 1 API call |

### Key design decisions

**NaN pre-filling** — AutoDDG's `analyze_semantics()` uses the `beartype` library for strict type checking and requires all DataFrame values to be strings. Government datasets frequently contain `NaN` values (pandas float). Passing these directly crashes the profiler with a type violation. Fixed by running `df.fillna("").astype(str)` before calling `analyze_semantics()`.

**Grouped semantic prompting** — `analyze_semantics()` is called with `use_group_prompting=True, group_size=0` which sends all columns in a single API call rather than one call per column. This significantly reduces cost and latency for wide datasets (e.g. datasets with 110 or 249 columns).

**Graceful failure** — every stage (C through G) is wrapped in its own try/except. If structural profiling fails, the pipeline continues without a profile. If semantic analysis fails, the pipeline continues without semantics. Only a failure in the core `describe_dataset()` call marks the record as `generation_status = "error"`. This ensures maximum throughput even when individual stages fail.

**Step 1 metadata finally used** — `generate_topic()` is the first point in the pipeline where the metadata collected in Step 1 (title and original description) is passed to the LLM as input. This bridges the ingestion and generation stages.

### Outputs per dataset

Each successfully processed dataset produces **5 outputs**:
- `dataset_profile` — structural statistics text from datamart-profiler
- `semantic_profile` — LLM-generated column semantics
- `topic` — 2–3 word topic string (e.g. `'Los Angeles Crime Data'`)
- `generated_description` — user-focused description (~700 chars)
- `search_description` — keyword-rich search variant (~2,600 chars)


Results are saved locally as Parquet at `LOCAL_OUTPUT_PARQUET` then uploaded to HDFS for Spark to read in the evaluation step.

In [10]:
#add api key here 


In [11]:
# ── Imports ────────────────────────────────────────────────────
import os
import io
import pandas as pd
import tempfile
from pathlib import Path
from openai import OpenAI
from autoddg import AutoDDG


# ── Quick API test before full run ─────────────────────────────
client_test = OpenAI(api_key=OPENAI_API_KEY)
response = client_test.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "say hello"}],
    max_tokens=5
)
print("✅ API key works! Response:", response.choices[0].message.content)

# ── Pipeline ───────────────────────────────────────────────────
with tempfile.TemporaryDirectory() as tmpdir:
    local_input_parquet = copy_hdfs_parquet_to_local(HDFS_PREPARED, tmpdir)
    records_pd = pd.read_parquet(local_input_parquet)

    # ── REMOVE head() limit for full run ──────────────────────
    # records_pd = records_pd.head(8)   # ← comment this out for full 232

    client  = OpenAI(api_key=OPENAI_API_KEY)
    autoddg = AutoDDG(client=client, model_name=MODEL_NAME)

    output_rows = []
    total = len(records_pd)
    print(f"[INFO] Total prepared records to describe: {total}")

    for idx, row in records_pd.iterrows():
        dataset_id           = row.get("dataset_id")
        source               = row.get("source")
        title                = row.get("title")
        original_description = row.get("original_description")
        download_url         = row.get("download_url")
        landing_page_url     = row.get("landing_page_url")
        sample_csv           = row.get("sample_csv")

        print(f"\n[INFO] Processing {idx+1}/{total} | {source} | {dataset_id}")

        generated_description  = None
        search_description     = None
        topic                  = None
        dataset_profile_text   = None
        semantic_profile_text  = None
        generation_status      = "error"
        error_message          = None

        try:
            # ── Step A: Shrink the CSV sample ──────────────────
            shrunk_sample_csv = shrink_sample_csv(
                sample_csv=sample_csv,
                max_chars=MAX_CHARS,
                max_lines=MAX_LINES,
            )
            if shrunk_sample_csv is None:
                raise ValueError("missing_or_empty_sample_csv_after_shrink")

            # ── Step B: Convert CSV string → pandas DataFrame ──
            try:
                df = pd.read_csv(io.StringIO(shrunk_sample_csv))
                print(f"  [INFO] DataFrame shape: {df.shape}")
            except Exception as e:
                raise ValueError(f"csv_parse_failed: {e}")

            # ── Step C: Structural Profiling ───────────────────
            # Uses datamart-profiler — NO LLM cost
            # Computes missing values, distributions, data types
            try:
                dataset_profile_text, _ = autoddg.profile_dataframe(df)
                print(f"  [INFO] Profile generated ({len(dataset_profile_text)} chars)")
            except Exception as e:
                dataset_profile_text = None
                print(f"  [WARN] Profiling failed: {e}")

            # ── Step D: Semantic Column Analysis ───────────────
            # LLM infers what each column MEANS
            # df_clean handles NaN values that crash the profiler
            try:
                df_clean = df.fillna("").astype(str)  # fix for NaN crash
                semantic_profile_text = autoddg.analyze_semantics(
                    df_clean,
                    use_group_prompting=True,  # all columns in 1 API call
                    group_size=0,
                )
                print(f"  [INFO] Semantic profile generated ({len(semantic_profile_text)} chars)")
            except Exception as e:
                semantic_profile_text = None
                print(f"  [WARN] Semantic analysis failed: {e}")

            # ── Step E: Topic Generation ────────────────────────
            # THIS is where Step 1 metadata finally gets used!
            # title + original_description + CSV → 2-3 word topic
            try:
                topic = autoddg.generate_topic(
                    title=str(title) if title and str(title) != "nan" else "",
                    original_description=str(original_description)
                        if original_description and str(original_description) != "nan"
                        else None,
                    dataset_sample=shrunk_sample_csv,
                )
                print(f"  [INFO] Topic generated: '{topic}'")
            except Exception as e:
                topic = None
                print(f"  [WARN] Topic generation failed: {e}")

            # ── Step F: Full Description Generation ────────────
            # Now passes ALL context — profile + semantics + topic + CSV
            # This is the upgraded version of what you had before
            _, generated_description = autoddg.describe_dataset(
                dataset_sample=shrunk_sample_csv,
                dataset_profile=dataset_profile_text,
                use_profile=dataset_profile_text is not None,
                semantic_profile=semantic_profile_text,
                use_semantic_profile=semantic_profile_text is not None,
                data_topic=topic,
                use_topic=topic is not None,
            )
            print(f"  [INFO] Description generated ({len(generated_description)} chars)")

            # ── Step G: Search-Focused Description ─────────────
            # Keyword-rich variant for search indexing
            # This is what your proposal called "Search-Focused Descriptions"
            try:
                if topic:
                    _, search_description = autoddg.expand_description_for_search(
                        description=generated_description,
                        topic=topic,
                    )
                    print(f"  [INFO] Search description generated ({len(search_description)} chars)")
                else:
                    print(f"  [WARN] Skipping search description — no topic available")
            except Exception as e:
                search_description = None
                print(f"  [WARN] Search description failed: {e}")

            generation_status = "success"

        except Exception as e:
            error_message = str(e)
            print(f"  [ERROR] {source} | {dataset_id} | {error_message}")

        output_rows.append({
            "dataset_id":            None if pd.isna(dataset_id) else str(dataset_id),
            "source":                None if pd.isna(source) else str(source),
            "title":                 None if pd.isna(title) else str(title),
            "original_description":  None if pd.isna(original_description) else str(original_description),
            "download_url":          None if pd.isna(download_url) else str(download_url),
            "landing_page_url":      None if pd.isna(landing_page_url) else str(landing_page_url),
            "sample_csv":            sample_csv,
            "dataset_profile":       dataset_profile_text,
            "semantic_profile":      semantic_profile_text,
            "topic":                 topic,
            "generated_description": generated_description,
            "search_description":    search_description,
            "generation_status":     generation_status,
            "error_message":         error_message,
        })

        # ── Progress tracker ───────────────────────────────────
        success_so_far = sum(1 for r in output_rows if r["generation_status"] == "success")
        print(f"  [PROGRESS] {success_so_far}/{len(output_rows)} successful so far")

    # ── Save output locally ────────────────────────────────────
    out_df      = pd.DataFrame(output_rows)
    output_path = Path(LOCAL_OUTPUT_PARQUET)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(output_path, index=False)

    print(f"\n{'='*60}")
    print(f"PIPELINE COMPLETE")
    print(f"{'='*60}")
    print(f"Total processed     : {len(out_df)}")
    print(f"Descriptions        : {out_df['generated_description'].notna().sum()}")
    print(f"Search descriptions : {out_df['search_description'].notna().sum()}")
    print(f"Topics generated    : {out_df['topic'].notna().sum()}")
    print(f"Profiles generated  : {out_df['dataset_profile'].notna().sum()}")
    print(f"Semantic profiles   : {out_df['semantic_profile'].notna().sum()}")
    print(f"\nBy source and status:")
    print(out_df.groupby(["source", "generation_status"]).size().reset_index(name="count"))
    print(f"\nSample topics generated:")
    print(out_df[out_df["topic"].notna()][["source", "title", "topic"]].head(10).to_string(index=False))
    print(f"{'='*60}")

✅ API key works! Response: Hello! How can I
[CMD] hdfs dfs -get hdfs:///user/rbp5812_nyu_edu/pipeline/step3_prepared /tmp/tmphiryzhqu
[INFO] Total prepared records to describe: 236

[INFO] Processing 1/236 | nyc_open_data | qhkz-4dqm
  [INFO] DataFrame shape: (20, 7)
  [INFO] Profile generated (708 chars)
  [INFO] Semantic profile generated (896 chars)
  [INFO] Topic generated: 'Vehicle Travel Behavior'
  [INFO] Description generated (629 chars)
  [INFO] Search description generated (2542 chars)
  [PROGRESS] 1/1 successful so far

[INFO] Processing 2/236 | nyc_open_data | wgnh-qwsg
  [INFO] DataFrame shape: (20, 110)
  [INFO] Profile generated (11078 chars)
  [INFO] Semantic profile generated (6236 chars)
  [INFO] Topic generated: 'Urban Travel Behavior'
  [INFO] Description generated (682 chars)
  [INFO] Search description generated (2905 chars)
  [PROGRESS] 2/2 successful so far

[INFO] Processing 3/236 | nyc_open_data | naav-ygga
  [INFO] DataFrame shape: (20, 39)
  [INFO] Profile g

  [INFO] Description generated (715 chars)
  [INFO] Search description generated (2667 chars)
  [PROGRESS] 21/24 successful so far

[INFO] Processing 25/236 | nyc_open_data | 48wd-x25j
  [INFO] DataFrame shape: (20, 8)
  [INFO] Profile generated (663 chars)
  [INFO] Semantic profile generated (1164 chars)
  [INFO] Topic generated: 'Suspension Demographics Analysis'
  [INFO] Description generated (689 chars)
  [INFO] Search description generated (2676 chars)
  [PROGRESS] 22/25 successful so far

[INFO] Processing 26/236 | nyc_open_data | ezy6-djsf
  [INFO] DataFrame shape: (20, 9)
  [INFO] Profile generated (676 chars)
  [INFO] Semantic profile generated (1463 chars)
  [INFO] Topic generated: 'Construction Street Closures'
  [INFO] Description generated (620 chars)
  [INFO] Search description generated (2286 chars)
  [PROGRESS] 23/26 successful so far

[INFO] Processing 27/236 | nyc_open_data | irna-apch
  [INFO] DataFrame shape: (20, 9)
  [INFO] Profile generated (848 chars)
  [INFO] S

  [INFO] Semantic profile generated (1998 chars)
  [INFO] Topic generated: 'Class Size Analysis'
  [INFO] Description generated (721 chars)
  [INFO] Search description generated (2388 chars)
  [PROGRESS] 43/48 successful so far

[INFO] Processing 49/236 | nyc_open_data | inb9-u6nt
  [INFO] DataFrame shape: (20, 28)
  [INFO] Profile generated (2179 chars)
  [INFO] Semantic profile generated (3866 chars)
  [INFO] Topic generated: 'Package Locker Usage'
  [INFO] Description generated (702 chars)
  [INFO] Search description generated (2622 chars)
  [PROGRESS] 44/49 successful so far

[INFO] Processing 50/236 | nyc_open_data | 95zn-7w5f
  [INFO] DataFrame shape: (20, 8)
  [INFO] Profile generated (663 chars)
  [INFO] Semantic profile generated (1202 chars)
  [INFO] Topic generated: 'Urban Heat Resilience'
  [INFO] Description generated (709 chars)
  [INFO] Search description generated (2530 chars)
  [PROGRESS] 45/50 successful so far

[INFO] Processing 51/236 | nyc_open_data | dpq6-sy7w
  [

  [INFO] Semantic profile generated (833 chars)
  [INFO] Topic generated: 'Sidewalk Inspection Data'
  [INFO] Description generated (696 chars)
  [INFO] Search description generated (2388 chars)
  [PROGRESS] 65/72 successful so far

[INFO] Processing 73/236 | nyc_open_data | n5c8-x7x2
  [INFO] DataFrame shape: (20, 48)
  [INFO] Profile generated (2814 chars)
  [INFO] Semantic profile generated (2105 chars)
  [INFO] Topic generated: 'Vehicle Weight Monitoring'
  [INFO] Description generated (642 chars)
  [INFO] Search description generated (2517 chars)
  [PROGRESS] 66/73 successful so far

[INFO] Processing 74/236 | nyc_open_data | iwat-y983
  [INFO] DataFrame shape: (20, 9)
  [INFO] Profile generated (659 chars)
  [INFO] Semantic profile generated (1185 chars)
  [INFO] Topic generated: 'Preplacement Facility Statistics'
  [INFO] Description generated (661 chars)
  [INFO] Search description generated (2404 chars)
  [PROGRESS] 67/74 successful so far

[INFO] Processing 75/236 | nyc_open_

  [INFO] Description generated (766 chars)
  [INFO] Search description generated (2636 chars)
  [PROGRESS] 87/95 successful so far

[INFO] Processing 96/236 | nyc_open_data | u9wf-3gbt
  [INFO] DataFrame shape: (20, 14)
  [INFO] Profile generated (1254 chars)
  [INFO] Semantic profile generated (1831 chars)
  [INFO] Topic generated: 'Building Footprints NYC'
  [INFO] Description generated (702 chars)
  [INFO] Search description generated (2428 chars)
  [PROGRESS] 88/96 successful so far

[INFO] Processing 97/236 | nyc_open_data | 5zhs-2jue
  [INFO] DataFrame shape: (20, 16)
  [INFO] Profile generated (1408 chars)
  [INFO] Semantic profile generated (2026 chars)
  [INFO] Topic generated: 'Building Footprints NYC'
  [INFO] Description generated (661 chars)
  [INFO] Search description generated (2532 chars)
  [PROGRESS] 89/97 successful so far

[INFO] Processing 98/236 | nyc_open_data | q9w2-yi4x
  [INFO] DataFrame shape: (20, 7)
  [INFO] Profile generated (646 chars)
  [INFO] Semantic pr

...

Unmatched latitude columns: ['Y_lat']
Unmatched longitude columns: ['X_lon']


  [INFO] Search description generated (2496 chars)
  [PROGRESS] 173/196 successful so far

[INFO] Processing 197/236 | data_gov | https://data.cdc.gov/api/views/55yu-xksw
  [INFO] DataFrame shape: (20, 21)
  [INFO] Profile generated (1525 chars)
  [INFO] Semantic profile generated (2816 chars)
  [INFO] Topic generated: 'Heart Disease Mortality'
  [INFO] Description generated (665 chars)
  [INFO] Search description generated (2510 chars)
  [PROGRESS] 174/197 successful so far

[INFO] Processing 198/236 | data_gov | 9e9ce3da-64a6-431a-b1ec-a4ba001a2c53
  [INFO] DataFrame shape: (18, 4)
  [INFO] Profile generated (299 chars)
  [INFO] Semantic profile generated (605 chars)
  [INFO] Topic generated: 'ED Capacity Analysis'
  [INFO] Description generated (714 chars)
  [INFO] Search description generated (2430 chars)
  [PROGRESS] 175/198 successful so far

[INFO] Processing 199/236 | data_gov | https://data.cdc.gov/api/views/w9j2-ggv5
  [INFO] DataFrame shape: (20, 5)
  [INFO] Profile generate

  [INFO] Semantic profile generated (2068 chars)
  [INFO] Topic generated: 'Hate Crime Incidents'
  [INFO] Description generated (642 chars)
  [INFO] Search description generated (2522 chars)
  [PROGRESS] 192/219 successful so far

[INFO] Processing 220/236 | data_gov | https://data.wa.gov/api/views/769e-73q6
  [INFO] DataFrame shape: (20, 14)
  [INFO] Profile generated (1167 chars)
  [INFO] Semantic profile generated (2080 chars)
  [INFO] Topic generated: 'License Transfers Washington'
  [INFO] Description generated (673 chars)
  [INFO] Search description generated (2368 chars)
  [PROGRESS] 193/220 successful so far

[INFO] Processing 221/236 | data_gov | https://data.transportation.gov/api/views/8ect-6jqj
  [INFO] DataFrame shape: (20, 25)
  [INFO] Profile generated (2245 chars)
  [INFO] Semantic profile generated (2582 chars)
  [INFO] Topic generated: 'Vehicle Trajectory Data'
  [INFO] Description generated (626 chars)
  [INFO] Search description generated (2547 chars)
  [PROGRESS] 

Unmatched longitude columns: ['Long Idle (watts)']


  [INFO] Search description generated (2603 chars)
  [PROGRESS] 206/235 successful so far

[INFO] Processing 236/236 | data_gov | https://data.energystar.gov/api/views/rxdj-2c88
  [INFO] DataFrame shape: (20, 68)
  [INFO] Profile generated (6538 chars)
  [INFO] Semantic profile generated (7520 chars)
  [INFO] Topic generated: 'ENERGY STAR Computers'
  [INFO] Description generated (667 chars)
  [INFO] Search description generated (2626 chars)
  [PROGRESS] 207/236 successful so far

PIPELINE COMPLETE
Total processed     : 236
Descriptions        : 207
Search descriptions : 207
Topics generated    : 207
Profiles generated  : 207
Semantic profiles   : 207

By source and status:
          source generation_status  count
0       data_gov             error      9
1       data_gov           success     65
2  nyc_open_data             error     20
3  nyc_open_data           success    142

Sample topics generated:
       source                                        title                      t

---
## Step 8 — Evaluation Table Construction

We read the generated descriptions and enrich them with:
- **`has_original_description`**: whether the dataset had a human-written description to compare against
- **`original_description_len`**: character length of the original description
- **`generated_description_len`**: character length of the generated description

This evaluation table is written to HDFS at `step5_evaluation` for analysis.

In [12]:
# First copy local parquet to HDFS
import subprocess
subprocess.run([
    "hdfs", "dfs", "-put", "-f",
    LOCAL_OUTPUT_PARQUET,
    f"{HDFS_BASE}/step4_descriptions.parquet"
], check=True)

# Now read from HDFS
desc_df = spark.read.parquet(f"{HDFS_BASE}/step4_descriptions.parquet")

eval_df = (
    desc_df
    .withColumn(
        "has_original_description",
        F.when(
            F.col("original_description").isNotNull() &
            (F.trim(F.col("original_description")) != ""),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "original_description_len",
        F.length(F.coalesce(F.col("original_description"), F.lit("")))
    )
    .withColumn(
        "generated_description_len",
        F.length(F.coalesce(F.col("generated_description"), F.lit("")))
    )
    .withColumn(
        "search_description_len",
        F.length(F.coalesce(F.col("search_description"), F.lit("")))
    )
    .withColumn(
        "has_topic",
        F.when(
            F.col("topic").isNotNull() &
            (F.trim(F.col("topic")) != ""),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "has_profile",
        F.when(
            F.col("dataset_profile").isNotNull() &
            (F.trim(F.col("dataset_profile")) != ""),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "has_semantic_profile",
        F.when(
            F.col("semantic_profile").isNotNull() &
            (F.trim(F.col("semantic_profile")) != ""),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        "dataset_id", "source", "title",
        "has_original_description",
        "original_description", "original_description_len",
        "generated_description", "generated_description_len",
        "search_description", "search_description_len",
        "topic", "has_topic",
        "dataset_profile", "has_profile",
        "semantic_profile", "has_semantic_profile",
        "generation_status", "error_message",
    )
)

eval_df.write.mode("overwrite").parquet(HDFS_EVAL)

print("[INFO] Evaluation table written.")
print(f"[INFO] Output path: {HDFS_EVAL}")
print(f"[INFO] Row count: {eval_df.count()}")

print("\n── Generation status by source ───────────────────────────")
eval_df.groupBy("source", "generation_status").count().show(truncate=False)

print("\n── Original description coverage ─────────────────────────")
eval_df.groupBy("source", "has_original_description").count().show(truncate=False)

print("\n── Topic / Profile / Semantic coverage ───────────────────")
eval_df.groupBy("source", "has_topic", "has_profile", "has_semantic_profile").count().show(truncate=False)

print("\n── Description length stats ──────────────────────────────")
eval_df.select(
    "dataset_id", "source", "title",
    "original_description_len",
    "generated_description_len",
    "search_description_len",
).show(50, truncate=False)

print("\n── Sample topics generated ───────────────────────────────")
eval_df.filter(F.col("has_topic") == True).select(
    "source", "title", "topic"
).show(20, truncate=False)

print("\n── Sample: Generated vs Search description ───────────────")
results_pd = eval_df.toPandas()
successful = results_pd[results_pd["generation_status"] == "success"]

for _, row in successful.sample(n=min(3, len(successful)), random_state=42).iterrows():
    print("─" * 70)
    print(f"Dataset : {row['title']}")
    print(f"Source  : {row['source']}")
    print(f"Topic   : {row['topic']}")
    print()
    print(f"Generated Description:\n  {row['generated_description']}")
    print()
    if row.get("search_description"):
        print(f"Search Description:\n  {row['search_description'][:300]}...")
    print()

[INFO] Evaluation table written.
[INFO] Output path: hdfs:///user/rbp5812_nyu_edu/pipeline/step5_evaluation
[INFO] Row count: 236

── Generation status by source ───────────────────────────
+-------------+-----------------+-----+
|source       |generation_status|count|
+-------------+-----------------+-----+
|nyc_open_data|success          |142  |
|data_gov     |success          |65   |
|data_gov     |error            |9    |
|nyc_open_data|error            |20   |
+-------------+-----------------+-----+


── Original description coverage ─────────────────────────
+-------------+------------------------+-----+
|source       |has_original_description|count|
+-------------+------------------------+-----+
|nyc_open_data|false                   |5    |
|data_gov     |true                    |74   |
|nyc_open_data|true                    |157  |
+-------------+------------------------+-----+


── Topic / Profile / Semantic coverage ───────────────────
+-------------+---------+-----------+--

+-------------+---------------------------------------------------------------------------------------------------+--------------------------+
|source       |title                                                                                              |topic                     |
+-------------+---------------------------------------------------------------------------------------------------+--------------------------+
|nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                            |Vehicle Travel Behavior   |
|nyc_open_data|Citywide Mobility Survey - Trip 2024                                                               |Urban Travel Behavior     |
|nyc_open_data|Citywide Mobility Survey - Day 2024                                                                |Urban Travel Behavior     |
|nyc_open_data|Citywide Mobility Survey - Household 2024                                                          |Mobility Behavior Analysis|

In [13]:
results_pd = eval_df.toPandas()
successful = results_pd[results_pd["generation_status"] == "success"]

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Datasets requested           : {NYC_LIMIT + DATA_GOV_LIMIT}")
print(f"  NYC Open Data              : {NYC_LIMIT}")
print(f"  Data.gov                   : {DATA_GOV_LIMIT}")
print(f"Descriptions generated       : {len(successful)}")
print(f"Generation errors            : {len(results_pd) - len(successful)}")
print()
print(f"Topics generated             : {successful['topic'].notna().sum()}")
print(f"Structural profiles          : {successful['dataset_profile'].notna().sum()}")
print(f"Semantic profiles            : {successful['semantic_profile'].notna().sum()}")
print(f"Search descriptions          : {successful['search_description'].notna().sum()}")
print()
print("Generated description length stats (user-focused):")
print(successful["generated_description_len"].describe().round(1).to_string())
print()
print("Search description length stats (search-focused):")
print(successful["search_description_len"].describe().round(1).to_string())
print("=" * 60)

# ── Qualitative sample — now shows topic + both descriptions ───
print("\n" + "=" * 70)
print("QUALITATIVE EVALUATION SAMPLE")
print("=" * 70)

sample_n = min(5, len(successful))
for _, row in successful.sample(n=sample_n, random_state=42).iterrows():
    print(f"\nDataset : {row['title']}")
    print(f"Source  : {row['source']}")
    print(f"Topic   : {row['topic']}")
    print("─" * 70)

    if pd.notna(row.get("original_description")) and str(row["original_description"]).strip():
        orig = str(row["original_description"])[:300]
        print(f"Original Description:\n  {orig}...")
    else:
        print("Original Description: [None]")

    print(f"\nGenerated Description ({row['generated_description_len']} chars):")
    print(f"  {row['generated_description']}")

    if pd.notna(row.get("search_description")):
        print(f"\nSearch Description ({row['search_description_len']} chars):")
        print(f"  {str(row['search_description'])[:400]}...")

    print("=" * 70)

PIPELINE SUMMARY
Datasets requested           : 400
  NYC Open Data              : 200
  Data.gov                   : 200
Descriptions generated       : 207
Generation errors            : 29

Topics generated             : 207
Structural profiles          : 207
Semantic profiles            : 207
Search descriptions          : 207

Generated description length stats (user-focused):
count    207.0
mean     675.6
std       38.1
min      554.0
25%      651.5
50%      674.0
75%      701.0
max      785.0

Search description length stats (search-focused):
count     207.0
mean     2499.7
std       138.6
min      2174.0
25%      2402.0
50%      2498.0
75%      2595.5
max      2905.0

QUALITATIVE EVALUATION SAMPLE

Dataset : Hate Crime Incident (Open Data)
Source  : data_gov
Topic   : Hate Crime Data
──────────────────────────────────────────────────────────────────────
Original Description:
  <div><div>The Tempe Police Department prides itself in its continued efforts to reduce harm within the 

---
## Step 9 — Evaluation: Text Similarity Metrics

We evaluate the quality of generated descriptions against original human-written descriptions using three complementary metrics:

| Metric | What it measures |
|--------|-----------------|
| **ROUGE-1** | Unigram (word) overlap between generated and original |
| **ROUGE-2** | Bigram overlap — captures phrase-level similarity |
| **ROUGE-L** | Longest common subsequence — measures fluency |
| **METEOR** | Word overlap accounting for synonyms and stemming |
| **BERTScore** | Deep semantic similarity using contextual embeddings |

Only datasets that have **both** an original and a generated description are included in this comparison (`n = 227`).

In [14]:
# Install evaluation libraries
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "rouge-score", "bert-score", "nltk", "--quiet"])

import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.meteor_score import meteor_score

# Load results
results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")

# ── Filter to datasets that have both original and generated ───
comparable = results_pd[
    results_pd["original_description"].notna() &
    results_pd["generated_description"].notna() &
    (results_pd["original_description"].str.strip() != "") &
    (results_pd["generation_status"] == "success")
].copy()

print(f"Datasets with both original and generated: {len(comparable)}")

# ── ROUGE ──────────────────────────────────────────────────────
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge1_gen, rouge2_gen, rougeL_gen = [], [], []
rouge1_srch, rouge2_srch, rougeL_srch = [], [], []

for _, row in comparable.iterrows():
    # Score generated description vs original
    s = scorer.score(row["original_description"], row["generated_description"])
    rouge1_gen.append(s["rouge1"].fmeasure)
    rouge2_gen.append(s["rouge2"].fmeasure)
    rougeL_gen.append(s["rougeL"].fmeasure)

    # Score search description vs original (if available)
    if pd.notna(row.get("search_description")) and str(row["search_description"]).strip():
        s2 = scorer.score(row["original_description"], str(row["search_description"]))
        rouge1_srch.append(s2["rouge1"].fmeasure)
        rouge2_srch.append(s2["rouge2"].fmeasure)
        rougeL_srch.append(s2["rougeL"].fmeasure)

print("\n── ROUGE Scores ──────────────────────────────────────────")
print(f"{'Metric':<12} {'User Desc':>12} {'Search Desc':>12}")
print("-" * 38)
print(f"{'ROUGE-1':<12} {np.mean(rouge1_gen):>12.4f} {np.mean(rouge1_srch):>12.4f}")
print(f"{'ROUGE-2':<12} {np.mean(rouge2_gen):>12.4f} {np.mean(rouge2_srch):>12.4f}")
print(f"{'ROUGE-L':<12} {np.mean(rougeL_gen):>12.4f} {np.mean(rougeL_srch):>12.4f}")

# ── METEOR ─────────────────────────────────────────────────────
meteor_gen, meteor_srch = [], []

for _, row in comparable.iterrows():
    ref = row["original_description"].split()
    hyp = row["generated_description"].split()
    meteor_gen.append(meteor_score([ref], hyp))

    if pd.notna(row.get("search_description")) and str(row["search_description"]).strip():
        hyp2 = str(row["search_description"]).split()
        meteor_srch.append(meteor_score([ref], hyp2))

print(f"\n── METEOR Score ──────────────────────────────────────────")
print(f"{'Metric':<12} {'User Desc':>12} {'Search Desc':>12}")
print("-" * 38)
print(f"{'METEOR':<12} {np.mean(meteor_gen):>12.4f} {np.mean(meteor_srch):>12.4f}")

# Store scores on comparable
comparable["rouge1"] = rouge1_gen
comparable["rouge2"] = rouge2_gen
comparable["rougeL"] = rougeL_gen
comparable["meteor"] = meteor_gen

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/rbp5812_nyu_edu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/rbp5812_nyu_edu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Datasets with both original and generated: 203

── ROUGE Scores ──────────────────────────────────────────
Metric          User Desc  Search Desc
--------------------------------------
ROUGE-1            0.2438       0.1785
ROUGE-2            0.0436       0.0309
ROUGE-L            0.1435       0.0994

── METEOR Score ──────────────────────────────────────────
Metric          User Desc  Search Desc
--------------------------------------
METEOR             0.1636       0.1592


---
## Step 10 — Evaluation: Retrieval (NDCG@10)

We simulate a dataset search engine scenario. For each test query we check how many relevant keywords appear in each description, then compute **NDCG@10** (Normalized Discounted Cumulative Gain).

A higher NDCG means relevant datasets appear earlier in search results. We compare:
- **Generated** descriptions (our AutoDDG output)
- **Original** descriptions (human-written baseline)

Test queries are chosen to match known datasets in our collection.

In [15]:
import numpy as np
import pandas as pd

results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")

# ── Test queries ───────────────────────────────────────────────
test_queries = {
    "shooting incidents NYC"      : ["shooting", "offender", "victim", "crime"],
    "NYC parking violations"      : ["parking", "violation", "fiscal"],
    "flood sensor data"           : ["flood", "sensor", "floodnet"],
    "bicycle pedestrian counts"   : ["bicycle", "pedestrian", "count"],
    "COVID hospitalization rates" : ["covid", "hospitalization", "monthly"],
    "electric vehicle population" : ["electric", "vehicle", "population"],
    "chronic disease indicators"  : ["chronic", "disease", "behavioral"],
    "motor vehicle collisions"    : ["collision", "crash", "vehicle"],
}

def compute_ndcg(query_keywords, descriptions, k=10):
    scores = []
    for desc in descriptions:
        desc_lower = str(desc).lower()
        hit = sum(1 for kw in query_keywords if kw in desc_lower)
        scores.append(hit)
    ideal  = sorted(scores, reverse=True)[:k]
    actual = scores[:k]
    def dcg(rels):
        return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(rels))
    idcg = dcg(ideal)
    return dcg(actual) / idcg if idcg > 0 else 0.0

successful     = results_pd[results_pd["generation_status"] == "success"]
all_generated  = successful["generated_description"].tolist()
all_search     = successful["search_description"].dropna().tolist()
all_original   = results_pd[results_pd["original_description"].notna()]["original_description"].tolist()

print("── Retrieval Evaluation (NDCG@10) ───────────────────────────────────")
print(f"{'Query':<40} {'User Desc':>10} {'Search Desc':>12} {'Original':>10}")
print("-" * 74)

ndcg_gen, ndcg_srch, ndcg_orig = [], [], []
for query, keywords in test_queries.items():
    gen_ndcg  = compute_ndcg(keywords, all_generated)
    srch_ndcg = compute_ndcg(keywords, all_search)
    orig_ndcg = compute_ndcg(keywords, all_original)
    ndcg_gen.append(gen_ndcg)
    ndcg_srch.append(srch_ndcg)
    ndcg_orig.append(orig_ndcg)
    print(f"{query:<40} {gen_ndcg:>10.4f} {srch_ndcg:>12.4f} {orig_ndcg:>10.4f}")

print("-" * 74)
print(f"{'Average':<40} {np.mean(ndcg_gen):>10.4f} {np.mean(ndcg_srch):>12.4f} {np.mean(ndcg_orig):>10.4f}")
print()
print("Key insight: Search descriptions should score HIGHER than user descriptions")
print("on retrieval because they are keyword-rich by design.")

── Retrieval Evaluation (NDCG@10) ───────────────────────────────────
Query                                     User Desc  Search Desc   Original
--------------------------------------------------------------------------
shooting incidents NYC                       0.1995       0.2472     0.2896
NYC parking violations                       0.2933       0.2713     0.3076
flood sensor data                            0.0000       0.0000     0.0000
bicycle pedestrian counts                    0.1324       0.1974     0.0454
COVID hospitalization rates                  0.0000       0.0000     0.0000
electric vehicle population                  0.3450       0.4845     0.4150
chronic disease indicators                   0.0000       0.1310     0.0000
motor vehicle collisions                     0.2313       0.2188     0.2955
--------------------------------------------------------------------------
Average                                      0.1502       0.1938     0.1691

Key insight: Search

---
## Step 11 — Evaluation: BERTScore

BERTScore measures semantic similarity using contextual embeddings from a pretrained language model (`distilbert-base-uncased`). Unlike ROUGE/METEOR which rely on exact word matches, BERTScore captures meaning even when different words are used.

We use `batch_size=4` to avoid out-of-memory errors on the master node.

In [ ]:
# from bert_score import score as bert_score
# import numpy as np

# # ── BERTScore on user-focused descriptions ─────────────────────
# refs = comparable["original_description"].tolist()
# hyps = comparable["generated_description"].tolist()

# P, R, F1 = bert_score(
#     hyps, refs,
#     lang="en",
#     model_type="distilbert-base-uncased",
#     batch_size=4,
#     verbose=True
# )

# print("\n── BERTScore — User Description (distilbert) ─────────────")
# print(f"Precision : {P.mean().item():.4f}")
# print(f"Recall    : {R.mean().item():.4f}")
# print(f"F1        : {F1.mean().item():.4f}")

# comparable["bertscore_f1"] = F1.numpy()

# # ── BERTScore on search descriptions ──────────────────────────
# search_comparable = results_pd[
#     results_pd["original_description"].notna() &
#     results_pd["search_description"].notna() &
#     (results_pd["original_description"].str.strip() != "") &
#     (results_pd["generation_status"] == "success")
# ].copy()

# if len(search_comparable) > 0:
#     refs_s = search_comparable["original_description"].tolist()
#     hyps_s = search_comparable["search_description"].tolist()

#     P2, R2, F1_2 = bert_score(
#         hyps_s, refs_s,
#         lang="en",
#         model_type="distilbert-base-uncased",
#         batch_size=4,
#         verbose=True
#     )

#     print("\n── BERTScore — Search Description (distilbert) ───────────")
#     print(f"Precision : {P2.mean().item():.4f}")
#     print(f"Recall    : {R2.mean().item():.4f}")
#     print(f"F1        : {F1_2.mean().item():.4f}")

# # ── Clean final summary — NO per-source breakdown ─────────────
# print("\n── Final Evaluation Summary ──────────────────────────────")
# print(f"{'Metric':<20} {'User Desc':>12} {'Search Desc':>14}")
# print("-" * 48)
# print(f"{'ROUGE-1':<20} {np.mean(rouge1_gen):>12.4f} {np.mean(rouge1_srch) if rouge1_srch else 0:>14.4f}")
# print(f"{'ROUGE-2':<20} {np.mean(rouge2_gen):>12.4f} {np.mean(rouge2_srch) if rouge2_srch else 0:>14.4f}")
# print(f"{'ROUGE-L':<20} {np.mean(rougeL_gen):>12.4f} {np.mean(rougeL_srch) if rougeL_srch else 0:>14.4f}")
# print(f"{'METEOR':<20} {np.mean(meteor_gen):>12.4f} {np.mean(meteor_srch) if meteor_srch else 0:>14.4f}")
# print(f"{'BERTScore F1':<20} {F1.mean().item():>12.4f} {F1_2.mean().item() if len(search_comparable) > 0 else 0:>14.4f}")
# print(f"{'NDCG@10':<20} {np.mean(ndcg_gen):>12.4f} {np.mean(ndcg_srch):>14.4f}")
# print("-" * 48)
# print()
# print("Note: Low ROUGE/METEOR is expected — generated descriptions")
# print("are data-grounded, not paraphrases of originals.")
# print("BERTScore is the most meaningful metric — it captures")
# print("semantic similarity regardless of exact wording.")
# print("Search descriptions optimized for retrieval, not similarity.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/102 [00:00<?, ?it/s]

computing greedy matching.


...

done in 23.39 seconds, 8.68 sentences/sec

── BERTScore — User Description (distilbert) ─────────────
Precision : 0.7660
Recall    : 0.7210
F1        : 0.7410


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/102 [00:00<?, ?it/s]

---
## Step 12 — Qualitative Evaluation

We randomly sample 5 successfully described datasets and display both the original and generated descriptions side by side for manual review.

Key aspects to assess:
- **Accuracy** — does the generated description correctly reflect the dataset content?
- **Readability** — is it clear and well-structured?
- **Informativeness** — does it add value over the original?

In [ ]:
# # ── Step 13: Complete Qualitative Evaluation ──────────────────
# import random
# import pandas as pd
# import numpy as np
# random.seed(42)

# # Load from local parquet — the raw output from Step 7
# results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")
# successful = results_pd[results_pd["generation_status"] == "success"].copy()

# # Compute all lengths on the fly
# # These don't exist in raw parquet — only in the Spark eval_df
# successful["generated_description_len"] = successful["generated_description"].fillna("").apply(len)
# successful["search_description_len"]    = successful["search_description"].fillna("").apply(len)
# successful["original_description_len"]  = successful["original_description"].fillna("").apply(len)

# # ── Pipeline Summary ───────────────────────────────────────────
# print("=" * 60)
# print("PIPELINE SUMMARY")
# print("=" * 60)
# print(f"Datasets requested           : {NYC_LIMIT + DATA_GOV_LIMIT}")
# print(f"  NYC Open Data              : {NYC_LIMIT}")
# print(f"  Data.gov                   : {DATA_GOV_LIMIT}")
# print(f"Descriptions generated       : {len(successful)}")
# print(f"Generation errors            : {len(results_pd) - len(successful)}")
# print()
# print(f"Topics generated             : {successful['topic'].notna().sum()}")
# print(f"Structural profiles          : {successful['dataset_profile'].notna().sum()}")
# print(f"Semantic profiles            : {successful['semantic_profile'].notna().sum()}")
# print(f"Search descriptions          : {successful['search_description'].notna().sum()}")
# print()
# print("User description length stats:")
# print(successful["generated_description_len"].describe().round(1).to_string())
# print()
# print("Search description length stats:")
# print(successful["search_description_len"].describe().round(1).to_string())
# print("=" * 60)

# # ── Qualitative Sample ─────────────────────────────────────────
# sample = successful.sample(n=min(5, len(successful)), random_state=42)

# print("\n" + "=" * 70)
# print("QUALITATIVE EVALUATION — FULL PIPELINE OUTPUT")
# print("=" * 70)

# for _, row in sample.iterrows():
#     print(f"\nDataset : {row['title']}")
#     print(f"Source  : {row['source']}")
#     print(f"Topic   : {row['topic']}")
#     print("─" * 70)

#     # Original description
#     if pd.notna(row.get("original_description")) and str(row["original_description"]).strip():
#         orig_len = len(str(row["original_description"]))
#         print(f"Original ({orig_len} chars):")
#         print(f"  {str(row['original_description'])[:300]}...")
#     else:
#         print("Original: [None — pipeline generated from scratch]")

#     # User-focused description
#     print(f"\nUser Description ({row['generated_description_len']} chars):")
#     print(f"  {row['generated_description']}")

#     # Search-focused description
#     if pd.notna(row.get("search_description")) and str(row["search_description"]).strip():
#         print(f"\nSearch Description ({row['search_description_len']} chars):")
#         print(f"  {str(row['search_description'])[:400]}...")
#     else:
#         print("\nSearch Description: [not generated]")

#     print("=" * 70)

# # ── Final Metrics Summary ──────────────────────────────────────
# print("\n" + "=" * 60)
# print("FINAL EVALUATION SUMMARY")
# print("=" * 60)

# # Reload comparable for metrics
# comparable = results_pd[
#     results_pd["original_description"].notna() &
#     results_pd["generated_description"].notna() &
#     (results_pd["original_description"].str.strip() != "") &
#     (results_pd["generation_status"] == "success")
# ].copy()

# print(f"Datasets used for evaluation : {len(comparable)}")
# print(f"  (have both original and generated description)")
# print()

# # ROUGE
# from rouge_score import rouge_scorer
# scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# rouge1_gen, rouge2_gen, rougeL_gen   = [], [], []
# rouge1_srch, rouge2_srch, rougeL_srch = [], [], []

# for _, row in comparable.iterrows():
#     s = scorer.score(row["original_description"], row["generated_description"])
#     rouge1_gen.append(s["rouge1"].fmeasure)
#     rouge2_gen.append(s["rouge2"].fmeasure)
#     rougeL_gen.append(s["rougeL"].fmeasure)

#     if pd.notna(row.get("search_description")) and str(row["search_description"]).strip():
#         s2 = scorer.score(row["original_description"], str(row["search_description"]))
#         rouge1_srch.append(s2["rouge1"].fmeasure)
#         rouge2_srch.append(s2["rouge2"].fmeasure)
#         rougeL_srch.append(s2["rougeL"].fmeasure)

# # METEOR
# import nltk
# from nltk.translate.meteor_score import meteor_score as meteor
# meteor_gen, meteor_srch = [], []

# for _, row in comparable.iterrows():
#     ref = row["original_description"].split()
#     hyp = row["generated_description"].split()
#     meteor_gen.append(meteor([ref], hyp))

#     if pd.notna(row.get("search_description")) and str(row["search_description"]).strip():
#         hyp2 = str(row["search_description"]).split()
#         meteor_srch.append(meteor([ref], hyp2))

# # NDCG
# test_queries = {
#     "shooting incidents NYC"      : ["shooting", "offender", "victim", "crime"],
#     "NYC parking violations"      : ["parking", "violation", "fiscal"],
#     "flood sensor data"           : ["flood", "sensor", "floodnet"],
#     "bicycle pedestrian counts"   : ["bicycle", "pedestrian", "count"],
#     "COVID hospitalization rates" : ["covid", "hospitalization", "monthly"],
#     "electric vehicle population" : ["electric", "vehicle", "population"],
#     "chronic disease indicators"  : ["chronic", "disease", "behavioral"],
#     "motor vehicle collisions"    : ["collision", "crash", "vehicle"],
# }

# def compute_ndcg(query_keywords, descriptions, k=10):
#     scores = []
#     for desc in descriptions:
#         desc_lower = str(desc).lower()
#         hit = sum(1 for kw in query_keywords if kw in desc_lower)
#         scores.append(hit)
#     ideal  = sorted(scores, reverse=True)[:k]
#     actual = scores[:k]
#     def dcg(rels):
#         return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(rels))
#     idcg = dcg(ideal)
#     return dcg(actual) / idcg if idcg > 0 else 0.0

# all_generated = successful["generated_description"].tolist()
# all_search    = successful["search_description"].dropna().tolist()
# all_original  = results_pd[results_pd["original_description"].notna()]["original_description"].tolist()

# ndcg_gen, ndcg_srch, ndcg_orig = [], [], []
# for query, keywords in test_queries.items():
#     ndcg_gen.append(compute_ndcg(keywords, all_generated))
#     ndcg_srch.append(compute_ndcg(keywords, all_search))
#     ndcg_orig.append(compute_ndcg(keywords, all_original))

# # Print full summary table
# print(f"{'Metric':<20} {'User Desc':>12} {'Search Desc':>14} {'Original':>10}")
# print("-" * 58)
# print(f"{'ROUGE-1':<20} {np.mean(rouge1_gen):>12.4f} {np.mean(rouge1_srch) if rouge1_srch else 0:>14.4f} {'—':>10}")
# print(f"{'ROUGE-2':<20} {np.mean(rouge2_gen):>12.4f} {np.mean(rouge2_srch) if rouge2_srch else 0:>14.4f} {'—':>10}")
# print(f"{'ROUGE-L':<20} {np.mean(rougeL_gen):>12.4f} {np.mean(rougeL_srch) if rougeL_srch else 0:>14.4f} {'—':>10}")
# print(f"{'METEOR':<20} {np.mean(meteor_gen):>12.4f} {np.mean(meteor_srch) if meteor_srch else 0:>14.4f} {'—':>10}")
# print(f"{'NDCG@10':<20} {np.mean(ndcg_gen):>12.4f} {np.mean(ndcg_srch):>14.4f} {np.mean(ndcg_orig):>10.4f}")
# print("-" * 58)
# print()
# print("Notes:")
# print("  • Low ROUGE/METEOR is expected — descriptions are data-grounded,")
# print("    not paraphrases of originals.")
# # print("  • Search descriptions optimized for retrieval, not text similarity.")
# print("  • NDCG@10 compares all 3 types against original human descriptions.")
# print("=" * 60)

# # ── Per-dataset topic sample ───────────────────────────────────
# print("\n── Sample topics generated ───────────────────────────────")
# topic_sample = successful[successful["topic"].notna()][["source", "title", "topic"]].head(15)
# for _, row in topic_sample.iterrows():
#     print(f"  [{row['source'][:3].upper()}] {row['title'][:45]:<45} → {row['topic']}")

# print("\n── Pipeline Complete ─────────────────────────────────────")
# print(f"  Total descriptions  : {len(successful)}")
# print(f"  User descriptions   : {successful['generated_description'].notna().sum()}")
# print(f"  Search descriptions : {successful['search_description'].notna().sum()}")
# print(f"  Topics              : {successful['topic'].notna().sum()}")
# print(f"  Profiles            : {successful['dataset_profile'].notna().sum()}")
# print(f"  Semantic profiles   : {successful['semantic_profile'].notna().sum()}")

## Step 12 — AutoDDG Evaluation

We use AutoDDG's own built-in evaluation framework to assess description quality. This is more appropriate than ROUGE/METEOR for our task because those metrics compare against the original descriptions — which are often missing, broken, or extremely brief. AutoDDG's evaluators assess quality directly, without needing a clean reference.

We run three complementary evaluations:

### Part 1 — Intrinsic Quality Evaluation (`GPTEvaluator`)

Scores each generated description **independently** on three dimensions, with no reference description needed:

| Dimension | What it measures |
|---|---|
| Completeness | Does the description cover the key aspects of the dataset? |
| Conciseness | Is the description appropriately brief without losing information? |
| Readability | Is the description clear, fluent, and well-structured? |

This is particularly valuable for the datasets that had **no original description at all** — where reference-based metrics cannot be applied. We sample 10 datasets to keep API cost low.

### Part 2 — Pairwise Evaluation (`GPT4oMiniPairwiseEvaluator`)

GPT-4o-mini acts as a judge and directly compares two descriptions head-to-head:
A = our generated description
B = original human-written description
→ Judge picks which one is better

Only datasets with a substantive original description (> 50 characters) are included in this comparison. This is the most compelling evaluation because it answers the core question directly: **is our generated description better or worse than what already existed?**

### Part 3 — ELO Ratings

We treat the pairwise comparisons as a tournament and compute ELO ratings — the same system used to rank chess players:

- Both start at a baseline of **1500**
- Each win increases your rating, each loss decreases it
- The magnitude of change depends on the current rating gap
- **ELO > 1500 means stronger performance in head-to-head competition**

ELO gives an intuitive tournament-style summary of aggregate performance across all pairwise comparisons.


### Evaluators used

| Evaluator | Model | Purpose |
|---|---|---|
| `GPTEvaluator` | gpt-4o-mini | Intrinsic quality scoring — no reference needed |
| `GPT4oMiniPairwiseEvaluator` | gpt-4o-mini | Head-to-head comparison of generated vs original |

Both are part of the AutoDDG framework's built-in evaluation module (`autoddg.evaluation`).


In [4]:
# ── Step 1: Restart Spark ──────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ.pop("SPARK_SUBMIT_OPTS", None)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master yarn --deploy-mode client pyspark-shell"

spark = (
    SparkSession.builder
    .appName("autoddg_eval_reload")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
print("✅ Spark restarted")


HDFS_BASE            = "hdfs:///user/rbp5812_nyu_edu/pipeline"
LOCAL_OUTPUT_PARQUET = "/home/rbp5812_nyu_edu/outputs/descriptions.parquet"
HDFS_METADATA        = f"{HDFS_BASE}/step1_metadata"
HDFS_SAMPLES         = f"{HDFS_BASE}/step2_samples"
HDFS_PREPARED        = f"{HDFS_BASE}/step3_prepared"
HDFS_EVAL            = f"{HDFS_BASE}/step5_evaluation"
NYC_LIMIT            = 200
DATA_GOV_LIMIT       = 200
MODEL_NAME           = "gpt-4o-mini"

print("✅ Variables re-defined")

# ── Step 3: Load descriptions from local parquet ──────────────
import pandas as pd
import numpy as np
import random
random.seed(42)

results_pd = pd.read_parquet(LOCAL_OUTPUT_PARQUET)
successful = results_pd[results_pd["generation_status"] == "success"].copy()

# Compute lengths on the fly
successful["generated_description_len"] = successful["generated_description"].fillna("").apply(len)
successful["search_description_len"]    = successful["search_description"].fillna("").apply(len)
successful["original_description_len"]  = successful["original_description"].fillna("").apply(len)

print(f"✅ Loaded {len(successful)} successful descriptions from local parquet")
print(f"   Topics           : {successful['topic'].notna().sum()}")
print(f"   Profiles         : {successful['dataset_profile'].notna().sum()}")
print(f"   Semantic profiles: {successful['semantic_profile'].notna().sum()}")
print(f"   Search descs     : {successful['search_description'].notna().sum()}")

26/05/07 06:40:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ Spark restarted
✅ Variables re-defined
✅ Loaded 207 successful descriptions from local parquet
   Topics           : 207
   Profiles         : 207
   Semantic profiles: 207
   Search descs     : 207


In [5]:
import pandas as pd
import numpy as np
import random
from autoddg import GPTEvaluator, GPT4oMiniPairwiseEvaluator, AutoDDG
from openai import OpenAI

random.seed(42)

# ── Setup ──────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)

# Single description evaluator — scores quality intrinsically
single_evaluator = GPTEvaluator(
    gpt4_api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini"       # use mini to save cost
)

# Pairwise evaluator — compares generated vs original
pairwise_evaluator = GPT4oMiniPairwiseEvaluator(
    gpt4_api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini"
)

# ── Load data ──────────────────────────────────────────────────
results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")
successful = results_pd[results_pd["generation_status"] == "success"].copy()

# For pairwise: only datasets that have BOTH original and generated
comparable = successful[
    successful["original_description"].notna() &
    successful["generated_description"].notna() &
    (successful["original_description"].str.strip() != "") &
    (successful["original_description"].str.len() > 50)  # skip very short originals
].copy().reset_index(drop=True)

print(f"Total successful        : {len(successful)}")
print(f"Usable for pairwise eval: {len(comparable)}")

# ── PART 1: Single Description Evaluation ─────────────────────
# Scores each generated description intrinsically
# No reference description needed — great for datasets with no original
print("\n" + "=" * 60)
print("PART 1: INTRINSIC QUALITY EVALUATION (GPTEvaluator)")
print("=" * 60)
print("Evaluates each generated description independently")
print("on quality dimensions — no reference description needed")
print()

# Sample 10 to keep cost low
single_sample = successful.sample(n=min(10, len(successful)), random_state=42)
single_results = []

for idx, (_, row) in enumerate(single_sample.iterrows()):
    print(f"[{idx+1}/10] Evaluating: {row['title'][:50]}")
    try:
        score_text = single_evaluator.evaluate(row["generated_description"])
        single_results.append({
            "title":       row["title"],
            "source":      row["source"],
            "topic":       row["topic"],
            "score_text":  score_text,
        })
        print(f"         Score: {score_text}")
    except Exception as e:
        print(f"         [WARN] Failed: {e}")

print(f"\n✅ Single evaluation complete: {len(single_results)}/10 scored")

# ── PART 2: Pairwise Evaluation — Generated vs Original ───────
# The most compelling evaluation for the presentation
# GPT-4o-mini directly judges which description is better
print("\n" + "=" * 60)
print("PART 2: PAIRWISE EVALUATION (GPT4oMiniPairwiseEvaluator)")
print("=" * 60)
print("Directly compares generated description vs original")
print("A = generated description wins | B = original wins")
print()

# Sample 20 pairs to keep cost manageable
pairwise_sample = comparable.sample(
    n=min(20, len(comparable)),
    random_state=42
).reset_index(drop=True)

pairwise_results = []
gen_wins  = 0
orig_wins = 0

for idx, row in pairwise_sample.iterrows():
    print(f"[{idx+1}/{len(pairwise_sample)}] {row['title'][:50]}")
    try:
        # A = generated, B = original
        winner = pairwise_evaluator.compare(
            desc_a=row["generated_description"],   # A = our output
            desc_b=row["original_description"],    # B = human original
        )
        result = "Generated" if winner == "A" else "Original"
        if winner == "A":
            gen_wins += 1
        else:
            orig_wins += 1

        pairwise_results.append({
            "title":        row["title"],
            "source":       row["source"],
            "topic":        row["topic"],
            "winner":       result,
            "generated":    row["generated_description"],
            "original":     row["original_description"],
        })
        print(f"         Winner: {result} ({'✅ our output' if result == 'Generated' else '❌ original'})")

    except Exception as e:
        print(f"         [WARN] Failed: {e}")

total_compared = gen_wins + orig_wins
print(f"\n── Pairwise Results ──────────────────────────────────────")
print(f"Total pairs compared  : {total_compared}")
print(f"Generated wins        : {gen_wins} ({gen_wins/total_compared*100:.1f}%)")
print(f"Original wins         : {orig_wins} ({orig_wins/total_compared*100:.1f}%)")

# ── PART 3: ELO Ratings ────────────────────────────────────────
# Treat generated vs original as a tournament
# ELO > 1500 means generated descriptions are stronger overall
print("\n" + "=" * 60)
print("PART 3: ELO RATINGS")
print("=" * 60)

if len(pairwise_results) >= 5:
    # Build descriptions list: [generated_0, original_0, generated_1, original_1...]
    all_descs = []
    elo_pairs_results = []

    for i, r in enumerate(pairwise_results):
        gen_idx  = i * 2
        orig_idx = i * 2 + 1
        all_descs.append(r["generated"])
        all_descs.append(r["original"])
        winner_idx = "A" if r["winner"] == "Generated" else "B"
        elo_pairs_results.append((gen_idx, orig_idx, winner_idx))

    elo_input = {"overall": elo_pairs_results}
    elo_ratings = pairwise_evaluator.compute_elo_ratings(
        elo_input,
        n_desc=len(all_descs)
    )

    gen_elos  = [elo_ratings["overall"][i*2]   for i in range(len(pairwise_results))]
    orig_elos = [elo_ratings["overall"][i*2+1] for i in range(len(pairwise_results))]

    print(f"Average ELO — Generated descriptions : {np.mean(gen_elos):.1f}")
    print(f"Average ELO — Original descriptions  : {np.mean(orig_elos):.1f}")
    print(f"(Baseline ELO = 1500 for both)")
    print()
    if np.mean(gen_elos) > np.mean(orig_elos):
        print("✅ Generated descriptions have HIGHER ELO than originals")
    else:
        print("⚠️  Original descriptions have higher ELO")
    print("Note: ELO > 1500 means stronger performance in head-to-head")

# ── PART 4: Detailed Breakdown ─────────────────────────────────
print("\n" + "=" * 60)
print("PART 4: DETAILED PAIRWISE BREAKDOWN")
print("=" * 60)

pairwise_df = pd.DataFrame(pairwise_results)
if len(pairwise_df) > 0:
    print("\nBy source:")
    print(pairwise_df.groupby(["source", "winner"]).size().unstack(fill_value=0))

    print("\nExamples where Generated WON:")
    gen_won = pairwise_df[pairwise_df["winner"] == "Generated"].head(2)
    for _, row in gen_won.iterrows():
        print(f"\n  Dataset : {row['title']}")
        print(f"  Topic   : {row['topic']}")
        print(f"  Original: {str(row['original'])[:150]}...")
        print(f"  Generated: {str(row['generated'])[:150]}...")

    print("\nExamples where Original WON:")
    orig_won = pairwise_df[pairwise_df["winner"] == "Original"].head(2)
    for _, row in orig_won.iterrows():
        print(f"\n  Dataset : {row['title']}")
        print(f"  Topic   : {row['topic']}")
        print(f"  Original: {str(row['original'])[:150]}...")
        print(f"  Generated: {str(row['generated'])[:150]}...")

# ── PART 5: Final Summary ──────────────────────────────────────
print("\n" + "=" * 60)
print("AUTODDG EVALUATION FINAL SUMMARY")
print("=" * 60)
print(f"Single evaluation (intrinsic quality) : {len(single_results)} descriptions scored")
print(f"Pairwise evaluation (head-to-head)    : {total_compared} pairs compared")
print(f"  Generated wins : {gen_wins} ({gen_wins/total_compared*100:.1f}%)")
print(f"  Original wins  : {orig_wins} ({orig_wins/total_compared*100:.1f}%)")
if len(pairwise_results) >= 5:
    print(f"  ELO Generated  : {np.mean(gen_elos):.1f}")
    print(f"  ELO Original   : {np.mean(orig_elos):.1f}")
print("=" * 60)

Total successful        : 207
Usable for pairwise eval: 201

PART 1: INTRINSIC QUALITY EVALUATION (GPTEvaluator)
Evaluates each generated description independently
on quality dimensions — no reference description needed

[1/10] Evaluating: Hate Crime Incident (Open Data)
         Score: Completeness: 6, Conciseness: 8, Readability: 7
[2/10] Evaluating: FloodNet: Sensor Deployment Metadata
         Score: Completeness: 8, Conciseness: 9, Readability: 8
[3/10] Evaluating: Digital Tax Map: Condominiums
         Score: Completeness: 7, Conciseness: 8, Readability: 7
[4/10] Evaluating: TAX_LOT_FACE
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[5/10] Evaluating: Local Area Unemployment Statistics (LAUS), Annual 
         Score: Completeness: 8, Conciseness: 9, Readability: 8
[6/10] Evaluating: Parking Violations Issued - Fiscal Year 2024
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[7/10] Evaluating: POSSESSION_HOOK
         Score: Completeness: 7, Conci

In [6]:
import pandas as pd
import numpy as np
import random
from autoddg import GPTEvaluator, GPT4oMiniPairwiseEvaluator
from openai import OpenAI

random.seed(42)

client = OpenAI(api_key=OPENAI_API_KEY)

single_evaluator = GPTEvaluator(
    gpt4_api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini"
)

pairwise_evaluator = GPT4oMiniPairwiseEvaluator(
    gpt4_api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini"
)

results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")
successful = results_pd[results_pd["generation_status"] == "success"].copy()

comparable = successful[
    successful["original_description"].notna() &
    successful["generated_description"].notna() &
    (successful["original_description"].str.strip() != "") &
    (successful["original_description"].str.len() > 50)
].copy().reset_index(drop=True)

print(f"Total successful        : {len(successful)}")
print(f"Usable for pairwise eval: {len(comparable)}")

# ── PART 1: Intrinsic Quality — all 207 ───────────────────────
print("\n" + "=" * 60)
print("PART 1: INTRINSIC QUALITY EVALUATION — ALL 207")
print("=" * 60)

single_results = []
completeness_scores = []
conciseness_scores  = []
readability_scores  = []

for idx, (_, row) in enumerate(successful.iterrows()):
    print(f"[{idx+1}/{len(successful)}] {row['title'][:55]}")
    try:
        score_text = single_evaluator.evaluate(row["generated_description"])
        single_results.append({
            "title":      row["title"],
            "source":     row["source"],
            "topic":      row["topic"],
            "score_text": score_text,
        })

        # Parse scores out of "Completeness: 7, Conciseness: 8, Readability: 8"
        import re
        nums = re.findall(r'\d+', score_text)
        if len(nums) >= 3:
            completeness_scores.append(int(nums[0]))
            conciseness_scores.append(int(nums[1]))
            readability_scores.append(int(nums[2]))

        print(f"         Score: {score_text}")
    except Exception as e:
        print(f"         [WARN] Failed: {e}")

print(f"\n✅ Intrinsic evaluation complete: {len(single_results)}/{len(successful)}")
if completeness_scores:
    print(f"\n── Average Scores (out of 10) ────────────────────────────")
    print(f"Completeness : {np.mean(completeness_scores):.2f}")
    print(f"Conciseness  : {np.mean(conciseness_scores):.2f}")
    print(f"Readability  : {np.mean(readability_scores):.2f}")
    print(f"Overall avg  : {np.mean(completeness_scores + conciseness_scores + readability_scores):.2f}")

# ── PART 2: Pairwise — ALL 201 comparable pairs ───────────────
print("\n" + "=" * 60)
print("PART 2: PAIRWISE EVALUATION — ALL 201 PAIRS")
print("=" * 60)
print("A = generated wins | B = original wins")
print()

# Use ALL comparable pairs — not just 20
pairwise_results = []
gen_wins  = 0
orig_wins = 0

for idx, row in comparable.iterrows():
    print(f"[{idx+1}/{len(comparable)}] {row['title'][:55]}")
    try:
        winner = pairwise_evaluator.compare(
            desc_a=row["generated_description"],
            desc_b=row["original_description"],
        )
        result = "Generated" if winner == "A" else "Original"
        if winner == "A":
            gen_wins += 1
        else:
            orig_wins += 1

        pairwise_results.append({
            "title":    row["title"],
            "source":   row["source"],
            "topic":    row["topic"],
            "winner":   result,
            "generated":row["generated_description"],
            "original": row["original_description"],
        })
        print(f"         Winner: {result} ({'✅' if result == 'Generated' else '❌'})")

    except Exception as e:
        print(f"         [WARN] Failed: {e}")

    # ── Save progress every 20 datasets in case kernel dies ───
    if (idx + 1) % 20 == 0:
        progress_df = pd.DataFrame(pairwise_results)
        progress_df.to_parquet(
            f"/home/rbp5812_nyu_edu/outputs/pairwise_progress_{idx+1}.parquet",
            index=False
        )
        print(f"\n  💾 Progress saved at {idx+1} pairs — "
              f"Gen: {gen_wins} / Orig: {orig_wins}\n")

total_compared = gen_wins + orig_wins
print(f"\n── Pairwise Results ──────────────────────────────────────")
print(f"Total pairs compared : {total_compared}")
print(f"Generated wins       : {gen_wins} ({gen_wins/total_compared*100:.1f}%)")
print(f"Original wins        : {orig_wins} ({orig_wins/total_compared*100:.1f}%)")

# ── PART 3: ELO ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART 3: ELO RATINGS")
print("=" * 60)

all_descs        = []
elo_pairs_results = []

for i, r in enumerate(pairwise_results):
    gen_idx  = i * 2
    orig_idx = i * 2 + 1
    all_descs.append(r["generated"])
    all_descs.append(r["original"])
    winner_idx = "A" if r["winner"] == "Generated" else "B"
    elo_pairs_results.append((gen_idx, orig_idx, winner_idx))

elo_input  = {"overall": elo_pairs_results}
elo_ratings = pairwise_evaluator.compute_elo_ratings(
    elo_input,
    n_desc=len(all_descs)
)

gen_elos  = [elo_ratings["overall"][i*2]   for i in range(len(pairwise_results))]
orig_elos = [elo_ratings["overall"][i*2+1] for i in range(len(pairwise_results))]

print(f"Generated ELO : {np.mean(gen_elos):.1f}")
print(f"Original ELO  : {np.mean(orig_elos):.1f}")
print(f"Baseline      : 1500")
print()
if np.mean(gen_elos) > np.mean(orig_elos):
    print("✅ Generated descriptions have HIGHER ELO than originals")

# ── PART 4: Breakdown by source ────────────────────────────────
print("\n" + "=" * 60)
print("PART 4: BREAKDOWN BY SOURCE")
print("=" * 60)

pairwise_df = pd.DataFrame(pairwise_results)
print(pairwise_df.groupby(["source", "winner"]).size().unstack(fill_value=0))

# ── PART 5: Save final results ─────────────────────────────────
pairwise_df.to_parquet(
    "/home/rbp5812_nyu_edu/outputs/pairwise_final_all.parquet",
    index=False
)
print("\n💾 Full pairwise results saved to pairwise_final_all.parquet")

# ── PART 6: Final Summary ──────────────────────────────────────
print("\n" + "=" * 60)
print("AUTODDG EVALUATION FINAL SUMMARY — FULL RUN")
print("=" * 60)
print(f"Datasets evaluated (intrinsic) : {len(single_results)}")
if completeness_scores:
    print(f"  Avg Completeness : {np.mean(completeness_scores):.2f} / 10")
    print(f"  Avg Conciseness  : {np.mean(conciseness_scores):.2f} / 10")
    print(f"  Avg Readability  : {np.mean(readability_scores):.2f} / 10")
print(f"\nPairwise evaluation            : {total_compared} pairs")
print(f"  Generated wins : {gen_wins} ({gen_wins/total_compared*100:.1f}%)")
print(f"  Original wins  : {orig_wins} ({orig_wins/total_compared*100:.1f}%)")
print(f"  ELO Generated  : {np.mean(gen_elos):.1f}")
print(f"  ELO Original   : {np.mean(orig_elos):.1f}")
print("=" * 60)

Total successful        : 207
Usable for pairwise eval: 201

PART 1: INTRINSIC QUALITY EVALUATION — ALL 207
[1/207] Citywide Mobility Survey - Vehicle 2024
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[2/207] Citywide Mobility Survey - Trip 2024
         Score: Completeness: 8, Conciseness: 8, Readability: 8
[3/207] Citywide Mobility Survey - Day 2024
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[4/207] Citywide Mobility Survey - Household 2024
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[5/207] Citywide Mobility Survey - Person 2024
         Score: Completeness: 8, Conciseness: 8, Readability: 8
[6/207] Shooting Offenders (2006-Present)
         Score: Completeness: 6, Conciseness: 8, Readability: 7
[7/207] Shooting Victims (2006-Present)
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[8/207] Shootings (2006-Present)
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[9/207] Parking Violations Issu

         Score: Completeness: 7, Conciseness: 8, Readability: 7
[72/207] Digital Tax Map: Condominium Units
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[73/207] Digital Tax Map: Air Lots
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[74/207] Digital Tax Map: Condominiums
         Score: Completeness: 7, Conciseness: 8, Readability: 7
[75/207] Digital Tax Map: Subterranean Lots
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[76/207] City Council Capital Budget
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[77/207] TLC Vehicles Involved in Crashes (Local Law 31)
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[78/207] Youth Count
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[79/207] NYC Honorary Street Names Map (Intersection)
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[80/207] NYC Drinking Water Tank Inspections and Audits Complian
         Score: Completeness

         Score: Completeness: 8, Conciseness: 8, Readability: 9
[149/207] Supply Chain Greenhouse Gas Emission Factors v1.3 by NA
         Score: Completeness: 8, Conciseness: 9, Readability: 8
[150/207] Lottery Powerball Winning Numbers: Beginning 2010
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[151/207] NCHS - Leading Causes of Death: United States
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[152/207] Monthly Rates of Laboratory-Confirmed COVID-19 Hospital
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[153/207] Mental Health Care in the Last 4 Weeks
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[154/207] Crimes - 2001 to Present
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[155/207] MTA Bus Fare Evasion: Beginning 2019
         Score: Completeness: 7, Conciseness: 8, Readability: 8
[156/207] Provisional COVID-19 death counts, rates, and percent o
         Score: Completeness: 7, Concisenes

         Winner: Generated (✅)
[15/201] Historical Voter Turnout
         Winner: Generated (✅)
[16/201] FloodNet: Sensor Deployment Metadata
         Winner: Generated (✅)
[17/201] NYC Planimetric Database: Cooling Towers
         Winner: Generated (✅)
[18/201] Suspension Report NYPD Contacts (2015-2018)
         Winner: Generated (✅)
[19/201] Class Size Report (2006-2007)
         Winner: Generated (✅)
[20/201] Advertised Lotteries on Housing Connect by Lottery
         Winner: Generated (✅)

  💾 Progress saved at 20 pairs — Gen: 19 / Orig: 1

[21/201] Advertised Lotteries on Housing Connect By Building
         Winner: Generated (✅)
[22/201] Multiple Removals 2015-2018
         Winner: Generated (✅)
[23/201] Street Closures due to Construction Activities by Block
         Winner: Generated (✅)
[24/201] 2024 Survey of Volunteerism
         Winner: Generated (✅)
[25/201] Attendance Results 2013-2019
         Winner: Generated (✅)
[26/201] Average Class Size 2017-2019 (Preliminary)
   

         Winner: Generated (✅)
[117/201] NYC Open Data Plan: FOIL Datasets
         Winner: Generated (✅)
[118/201] NYC Open Data Plan: FOIL Metrics
         Winner: Generated (✅)
[119/201] NYC Open Data Plan: Removals
         Winner: Generated (✅)
[120/201] NYC Open Data Plan: Future Releases
         Winner: Generated (✅)

  💾 Progress saved at 120 pairs — Gen: 114 / Orig: 6

[121/201] Bicycle Parking
         Winner: Generated (✅)
[122/201] New York City Automated External Defibrillator (AED) In
         Winner: Generated (✅)
[123/201] VZV Workshops Locations
         Winner: Generated (✅)
[124/201] VZV Town Hall Locations
         Winner: Generated (✅)
[125/201] VZV Taxi & Car Service Trainings
         Winner: Generated (✅)
[126/201] VZV Street Team Flyers
         Winner: Generated (✅)
[127/201] VZV Street Improvement Projects (SIP) Intersections
         Winner: Generated (✅)
[128/201] VZV Senior Centers
         Winner: Generated (✅)
[129/201] VZV Priority Intersections
      

---
## Pipeline Complete

All steps have been executed successfully. Below is a summary of outputs:

| Output | Location |
|--------|----------|
| Metadata | `hdfs:///user/rbp5812_nyu_edu/pipeline/step1_metadata` |
| CSV Samples | `hdfs:///user/rbp5812_nyu_edu/pipeline/step2_samples` |
| Prepared Data | `hdfs:///user/rbp5812_nyu_edu/pipeline/step3_prepared` |
| Generated Descriptions | `/home/rbp5812_nyu_edu/outputs/descriptions.parquet` |

Stopping the Spark session releases cluster resources.

In [ ]:
spark.stop()
print("Spark session stopped.")
print(f"Final descriptions : {LOCAL_OUTPUT_PARQUET}")
print(f"Evaluation table   : {HDFS_EVAL}")
